In [1]:
import pandas as pd
from pathlib import Path
from collections import Counter
import unicodedata
import os
import re
import zlib
import fitz
import olefile

In [2]:
import sys
print(sys.executable)  # 현재 '파이썬'이 실행되고 있는 경로 (myenv여야 함)

!which pip  # '!' 명령어가 바라보는 pip의 실제 경로 (/opt/jhub-venv/... 일 확률 높음)

/home/spai0722/myenv/bin/python
/opt/jhub-venv/bin/pip


In [3]:
import torch
print(torch.cuda.is_available())

True


In [4]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)

## (0)흐름

- hwp->pdf 파일명/파일형식 수정<br>
- pdf 재파싱 및 노이즈 제거<br>
- hwp 재파싱 및 노이즈 제거<br>
- pdf,hwp concat<br>
- 결측처리

## (1)데이터 불러오기

In [5]:
df = pd.read_csv("/home/shared/data_list.csv", encoding="utf-8")  
df.head(3)

,공고 번호,공고 차수,사업명,사업 금액,발주 기관,공개 일자,입찰 참여 시작일,입찰 참여 마감일,사업 요약,파일형식,파일명,텍스트
0,20241001798,0.0,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,130000000.0,한영대학,2024-10-04 13:51:23,NaN,2024-10-15 17:00:00,- 한영대학교 특성화 맞춤형 교육환경 구축을 위해 트랙운영 학사정보시스템을 고도화한...,hwp,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,\n \n2024년 특성화 맞춤형 교육환경 구축 – 트랙운영 학사정보시스템 ...
1,20241002912,0.0,2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선,129300000.0,한국연구재단,2024-10-04 15:01:52,2024-10-14 10:00:00,2024-10-16 14:00:00,- 사업 개요: 2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선\n...,hwp,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,\r\n \r\n \r\n \r\n제 안 요 청 서\r\n[ 2024년 대학 ...
2,20240827859,0.0,EIP3.0 고압가스 안전관리 시스템 구축 용역,40000000.0,한국생산기술연구원,2024-08-28 11:31:02,2024-08-29 09:00:00,2024-09-09 10:00:00,- 사업 개요: EIP3.0 고압가스 안전관리 시스템 구축 용역\n- 추진배경: 안...,hwp,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,\r\n \r\nEIP3.0 고압가스 안전관리\r\n시스템 구축 용역\...


In [6]:
df.columns

Index(['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일',
       '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '텍스트'],
      dtype='str')

In [7]:
df.shape

(100, 12)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   공고 번호      82 non-null     str    
 1   공고 차수      82 non-null     float64
 2   사업명        100 non-null    str    
 3   사업 금액      99 non-null     float64
 4   발주 기관      100 non-null    str    
 5   공개 일자      100 non-null    str    
 6   입찰 참여 시작일  74 non-null     str    
 7   입찰 참여 마감일  92 non-null     str    
 8   사업 요약      100 non-null    str    
 9   파일형식       100 non-null    str    
 10  파일명        100 non-null    str    
 11  텍스트        100 non-null    str    
dtypes: float64(2), str(10)
memory usage: 853.9 KB


### (1-1)hwp->pdf 관련 수정

In [9]:
folder = Path("/home/shared/files")  

ext_counter = Counter()

for f in folder.rglob("*"):
    if f.is_file():
        ext = f.suffix.lower() if f.suffix else "(no_extension)"
        ext_counter[ext] += 1

print("확장자별 개수")
for ext, count in sorted(ext_counter.items()):
    print(f"{ext}: {count}")

확장자별 개수
.docx: 1
.hwp: 94
.pdf: 6


In [10]:
df["파일형식"].value_counts() # 수정필요

파일형식
hwp    96
pdf     4
Name: count, dtype: int64

In [11]:
# hwp->pdf 변환하는 과정에서 파일명이 살짝 바뀌어 바뀐 이름으로 매핑
rename_map = {
    "한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp":
        "한국농어촌공사_아세안+3+식량안보정보시스템(AFSIS)+3단계+협력(캄보디아.hwp.pdf",
    "대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp":
        "대전대학교_대전대학교+2024학년도+다층적+융합+학습경험+플랫폼(MILE)+전.hwp.pdf"
}

df["파일명"] = df["파일명"].replace(rename_map)

In [12]:
# hwp->pdf로 파일형식 수정
targets = [
    "한국농어촌공사_아세안+3+식량안보정보시스템(AFSIS)+3단계+협력(캄보디아.hwp.pdf",
    "대전대학교_대전대학교+2024학년도+다층적+융합+학습경험+플랫폼(MILE)+전.hwp.pdf",
]

df.loc[df["파일명"].isin(targets), "파일형식"] = "pdf"

In [13]:
df.loc[df["파일명"].isin(targets), ["파일명", "파일형식"]]

,파일명,파일형식
25,한국농어촌공사_아세안+3+식량안보정보시스템(AFSIS)+3단계+협력(캄보디아.hwp...,pdf
42,대전대학교_대전대학교+2024학년도+다층적+융합+학습경험+플랫폼(MILE)+전.hw...,pdf


In [14]:
df["파일형식"].value_counts() # 파일형식 수정후

파일형식
hwp    94
pdf     6
Name: count, dtype: int64

#### 문서이름 비교

In [256]:
import pandas as pd
import unicodedata

def normalize_name(x):
    x = str(x).strip()
    x = unicodedata.normalize("NFC", x)   # 한글 정규화
    return x

# 메타데이터 파일명
meta_names = (
    df["파일명"]
    .dropna()
    .astype(str)
    .map(normalize_name)
)

# files 폴더 파일명
files_names = pd.Series(
    [normalize_name(f.name) for f in folder.rglob("*") if f.is_file()]
)

# set 비교
only_in_meta = sorted(set(meta_names) - set(files_names))
only_in_files = sorted(set(files_names) - set(meta_names))

print("메타데이터에만 있는 파일 수:", len(only_in_meta))
print("files 폴더에만 있는 파일 수:", len(only_in_files))

print("\n[메타데이터에만 있는 파일명]")
for x in only_in_meta:
    print(x)

print("\n[files 폴더에만 있는 파일명]")
for x in only_in_files:
    print(x)

메타데이터에만 있는 파일 수: 0
files 폴더에만 있는 파일 수: 1

[메타데이터에만 있는 파일명]

[files 폴더에만 있는 파일명]
고려대학교_차세대 포털·학사 정보시스템 구축사업.docx


### (1-2)pdf 재파싱 및 노이즈제거

In [257]:
# 텍스트 길이 <=500
short_rows = df.loc[df["텍스트"].fillna("").str.len() <= 500, ["파일명", "텍스트"]].copy()
short_rows["텍스트길이"] = short_rows["텍스트"].fillna("").str.len()

print(short_rows[["파일명", "텍스트길이"]]) # pdf 1개, hwp 6개

                                             파일명  텍스트길이
2       한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp    234
8      재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp    298
12    서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf    220
17  2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp    186
18   한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp     89
20        전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp    130
49     국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp    401


In [258]:
FILES_DIR = "/home/shared/files"  
file_name = "서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf"
pdf_file = os.path.join(FILES_DIR, file_name)

In [259]:
##### PDF에서 텍스트 추출하는 함수 #####
def extract_pdf_text(file_path):
    with fitz.open(file_path) as doc:
        return "".join(page.get_text() for page in doc)


##### 정제용 함수들 모음 #####
# ** 주석에 붙어있는것 : 직접 특정 단어 추가해서 커스텀한 함수
    
## 1.  띄어쓰기가 이상하게 깨진 단어도 찾아내기 위한 “검색 패턴”을 만드는 함수 
## 예시 : "제안요청서"를 찾되, PDF에서 제 안 요 청 서, 제안 요 청 서, 제 안요 청서 처럼 있어도 같은 말로 인식하게 만드는 함수
def build_spaced_phrase_pattern(phrase):
    chars = [re.escape(ch) for ch in phrase if ch.strip()]
    return r'(?<![가-힣A-Za-z0-9])' + r'\s*'.join(chars) + r'(?![가-힣A-Za-z0-9])' 

## 2. 허용한 문자 외의 문자를 공백으로 바꿈 (허용한 문자: 한글, 영문, 숫자, 공백, 일부 문장기호)
##  PDF 추출 시 섞여 들어오는 이상 문자, 제어문자, 깨진 특수기호를 줄이는 역할
def remove_unwanted_characters(text):
    pattern = r'[^가-힣a-zA-Z0-9\s\.\(\)\[\]\/\,\%\:\-\·\?\!\@]'
    cleaned_text, count = re.subn(pattern, ' ', text)
    return cleaned_text, count


## 3. 쪽번호 패턴 제거 (- 1 -, - 12 - 같은)
## 앞뒤가 한글/영문/숫자가 아닐 때만 지우도록 해서 일반 문장 속 하이픈은 덜 건드리게 만듬 (전화번호 등)
def remove_page_markers(text):
    pattern = r'(?<![\dA-Za-z가-힣])-\s*\d{1,3}\s*-(?![\dA-Za-z가-힣])'
    cleaned_text, count = re.subn(pattern, ' ', text)
    return cleaned_text, count

## 4. 반복적으로 붙는 문구를 제거
## 기본값은 한개만 넣어서 **커스텀 추가 가능. 문서마다 공통 노이즈가 더 보이면 리스트에 추가할 수 있게 만들어 둔 구조
def remove_repeated_markers(text, markers=None):
    if markers is None:
        markers = [
            "[사전공개용]",
        ]

    counts = {}
    cleaned_text = text

    for marker in markers:
        cleaned_text, count = re.subn(re.escape(marker), ' ', cleaned_text)
        counts[marker] = count

    return cleaned_text, counts

## 5. 공백 처리 함수. 한 줄짜리 선형 텍스트에 가깝게 안정화
def normalize_spacing_and_symbols(text):
    counts = {}

    text, count1 = re.subn(r'[\.·]{2,}', ' ', text)  # ......, ···· 같은 연속 점선을 공백으로 바꿉니다.
    counts["dot_leaders_removed"] = count1

    text, count2 = re.subn(r'[-_=]{3,}', ' ', text) # ---, ___, === 같은 반복 기호를 공백으로 바꿉니다.
    counts["repeated_symbols_removed"] = count2

    text, count3 = re.subn(r'\s+', ' ', text)  # 여러 공백, 줄바꿈, 탭 등을 하나의 공백으로 줄입니다
    counts["whitespace_normalized"] = count3

    text = text.strip() # 맨 앞뒤 공백을 없앱니다
    return text, counts


# 6. PDF 추출에서 자주 깨지는 핵심 용어를 사전 기반으로 복구
# 문서에서 중요하고 자주 나오는 표현만 안전하게 지정해서 복원. **커스텀 가능
def fix_spaced_phrases(text, phrases=None):
    if phrases is None:
        phrases = [
            "제안요청서",
            "사업명",
            "사업비",
            "사업기간",
            "제안안내",
            "제안서",
            "서울시립대학교",
            "입학처",
            "통합시스템",
            "학업성취도",
            "종단분석",
        ]

    counts = {}
    cleaned_text = text

    for phrase in phrases:
        pattern = build_spaced_phrase_pattern(phrase)
        cleaned_text, count = re.subn(pattern, phrase, cleaned_text)
        counts[phrase] = count

    return cleaned_text, counts


# 7. 문서 앞부분의 목차 제거
def remove_front_toc_block(text):
    report = {
        "toc_removed": False,
        "toc_removed_chars": 0,
    }

    toc_start = text.find("목 차") # "목 차" 위치를 찾고, "사업안내 1. 사업개요"가 두 번 이상 등장하는지 봅니다.
    body_matches = [m.start() for m in re.finditer(re.escape("사업안내 1. 사업개요"), text)]

    if toc_start != -1 and len(body_matches) >= 2: # 첫 번째는 목차, 두 번째는 실제 본문 시작이라고 가정합니다.
        real_body_start = body_matches[1]

        if toc_start < real_body_start and toc_start < 3000: # '목 차' 단어 위치가 문서 초반 3000자 안에 있을 때만 제거
            report["toc_removed"] = True
            report["toc_removed_chars"] = real_body_start - toc_start
            text = text[:toc_start].rstrip() + " " + text[real_body_start:].lstrip() # 양쪽 공백 정리

    return text, report


# 8. 어색한 표현 수동 치환. 잔여 예외 케이스 보정. **커스텀 가능
def fix_additional_phrases(text, replacements=None):
    if replacements is None:
        replacements = {
            "기개 발": "기개발",
            "시 스템": "시스템",
            "핵 심역량": "핵심역량",
            "학부 과": "학부과",
            "입시전 형": "입시전형",
            "인 프라": "인프라",
            "정 량적": "정량적",
            "프로파일 링": "프로파일링",
            "용 역 업 체": "용역업체",
            "작 성 지 침": "작성 지침"
        }

    counts = {}
    cleaned_text = text

    for old, new in replacements.items():
        cleaned_text, count = re.subn(re.escape(old), new, cleaned_text)
        counts[f"{old} -> {new}"] = count

    return cleaned_text, counts


##### 실제 실행해야하는 정제 함수 (위 모든 !!정제!! 함수 포함) #####
# 이상 문자를 먼저 정리해야 이후 정규식이 덜 흔들리고,
# 공백을 어느 정도 정리한 뒤에 핵심 용어 복원을 해야 패턴 매칭이 잘 되고,
# 목차 제거는 텍스트 구조가 어느 정도 정돈된 뒤 하는 편이 안전하므로 해당 순서로 진행
# 각 단계에서 치환된 횟수나 제거 여부를 report에 쌓습니다.
def clean_pdf_text_integrated(text):
    report = {
        "before_length": len(text), # 정제 전 시작 길이 기록
    }

    cleaned_text = unicodedata.normalize("NFC", text) # 한글 정규화

    cleaned_text, removed_char_count = remove_unwanted_characters(cleaned_text) # 불필요 문자 제거
    report["unwanted_characters_removed"] = removed_char_count

    cleaned_text, page_count = remove_page_markers(cleaned_text) # 페이지 번호 제거
    report["page_markers_removed"] = page_count

    cleaned_text, marker_counts = remove_repeated_markers(cleaned_text) # 반복 표식 제거**
    report["marker_removed_counts"] = marker_counts

    cleaned_text, spacing_counts = normalize_spacing_and_symbols(cleaned_text) # 공백/기호 정리
    report["spacing_symbol_counts"] = spacing_counts

    cleaned_text, phrase_counts = fix_spaced_phrases(cleaned_text) # 띄어진 핵심 용어 복원**
    report["spaced_phrase_fixed_counts"] = phrase_counts

    cleaned_text, toc_report = remove_front_toc_block(cleaned_text) # 앞쪽 목차 제거
    report["toc_report"] = toc_report

    cleaned_text, additional_phrase_counts = fix_additional_phrases(cleaned_text) # 추가 예외 표현 치환**
    report["additional_phrase_counts"] = additional_phrase_counts

    report["after_length"] = len(cleaned_text) # 정제 후 길이 기록
    return cleaned_text, report # 정제된 text, report 반환

In [261]:
##### 실제 실행 부분 #####
text = extract_pdf_text(pdf_file)  # PDF에서 원문 추출
text_cleaned, clean_report = clean_pdf_text_integrated(text)  # 추출한 원문에 순차적 정제 함수(총괄 함수)를 적용합니다.

mask = df["파일명"] == file_name # 전체 metadata csv에서 이 pdf 파일 찾기
# 문서명이 df에 없으면 즉시 에러
if not mask.any():
    raise ValueError(f"df에서 파일명을 찾지 못했습니다: {file_name}")

# df(metadata)의 텍스트 및 텍스트 길이 데이터 덮어쓰기
df.loc[mask, "텍스트"] = text_cleaned
df.loc[mask, "텍스트길이"] = len(text_cleaned)

# 정제 내역 출력
print("통합 정제 리포트")
for key, value in clean_report.items():
    print(f"- {key}: {value}")

print("\n원본 추출 길이:", len(text))
print("정제 후 길이:", len(text_cleaned))
print("줄어든 문자 수:", len(text) - len(text_cleaned))

preview_df = pd.DataFrame([
    {"구간": "앞부분", "text": text_cleaned[:1500]},
    {"구간": "중간부분", "text": text_cleaned[len(text_cleaned)//2: len(text_cleaned)//2 + 1500]},
    {"구간": "뒷부분", "text": text_cleaned[-1500:]},
])

display(preview_df) # preview_df는 앞/중간/뒤 1500자씩 잘라서 보여줌

# 앞 800자를 정제 전후로 직접 비교 출력
print("\n[정제 전 앞 800자]")
print(text[:800])
print("\n" + "=" * 100 + "\n")
print("[정제 후 앞 800자]")
print(text_cleaned[:800])

# 최종 정제본을 지정 위치에 .txt 파일로 저장
output_path_cleaned = Path("/home/bidcoin/서울시립대학교_pdf_노이즈제거_통합2.txt") 
output_path_cleaned.write_text(text_cleaned, encoding="utf-8")
print("\n저장 완료:", output_path_cleaned)

통합 정제 리포트
- before_length: 116224
- unwanted_characters_removed: 2155
- page_markers_removed: 146
- marker_removed_counts: {'[사전공개용]': 1}
- spacing_symbol_counts: {'dot_leaders_removed': 27, 'repeated_symbols_removed': 0, 'whitespace_normalized': 24570}
- spaced_phrase_fixed_counts: {'제안요청서': 11, '사업명': 13, '사업비': 2, '사업기간': 14, '제안안내': 2, '제안서': 72, '서울시립대학교': 6, '입학처': 15, '통합시스템': 31, '학업성취도': 24, '종단분석': 24}
- toc_report: {'toc_removed': True, 'toc_removed_chars': 403}
- additional_phrase_counts: {'기개 발 -> 기개발': 1, '시 스템 -> 시스템': 2, '핵 심역량 -> 핵심역량': 1, '학부 과 -> 학부과': 1, '입시전 형 -> 입시전형': 1, '인 프라 -> 인프라': 1, '정 량적 -> 정량적': 1, '프로파일 링 -> 프로파일링': 1, '용 역 업 체 -> 용역업체': 2, '작 성 지 침 -> 작성 지침': 1}
- after_length: 102589

원본 추출 길이: 116224
정제 후 길이: 102589
줄어든 문자 수: 13635


,구간,text
0,앞부분,제안요청서 본 제안요청서는 입찰참여의 균등한 기회 제공을 위해 규격을 공개하기 위한...
1,중간부분,1 0.5 3. 여성고용 우수기업(최근 3개월 평균 여성고용률과 최근 3개월 평균 ...
2,뒷부분,"에 기재한다. 2. 수행경험은 최근 3년간 유사사업 분야 실적 건수, 금액을 해당란..."



[정제 전 앞 800자]
[사전공개용]
제 안 요 청 서
본 제안요청서는 입찰참여의 균등한 기회 제공을 위해 
규격을 공개하기 위한 자료로써 실제 입찰공고 시 사업금액, 
과업내용, 평가항목, 제출서류 등은 변경될 수 있으니 반드시 
확인하시기 바랍니다.
2023. 06.
담당
성명
소 속
전화번호
e-mail
이석준
서울시립대학교
(입학처)
02-6490-6176
lsjptrs@uos.ac.kr
사 업 명
학업성취도 다차원 종단분석 통합시스템 1차 고도화
주관기관
서 울 시 립 대 학 교   입 학 처 
목       차
Ⅰ. 사업안내
  1. 사업개요 ······································································································ 01
  2. 추진배경 및 필요성 ················································································ 01
  3. 사업근거 및 3개년 추진계획 ······························································· 01
  4. 사업범위 ······································································································ 02
  5. 기대효과 ················································································


[정제 후 앞 800자]
제안요청서 본 제안요청서는 입찰참여의 균등한 기회 제공을 위해 규격을 공개하기 위한 자료로써 실제 입찰공고 시 사업금액, 과업내용, 평가항목, 제출서류 등은 변경될 수 있으니 반드시 확인하시기 바랍니다. 2023. 06. 담당 성명 소 속 전화번호 e-mail 이석준 서울시립대학교 (입학처) 02-6

In [262]:
# print("cwd:", Path.cwd())
# print("home:", Path.home())
# print("user:", os.getenv("USER"))

In [263]:
# [PDF 추출] df에서 PDF 행 전체를 df_pdf.csv로 저장
from pathlib import Path

pdf_mask = (
    df["파일명"].fillna("").str.lower().str.endswith(".pdf")
    | (df["파일형식"].fillna("").str.lower() == "pdf" if "파일형식" in df.columns else False)
)
df_pdf = df.loc[pdf_mask].copy()

out_path = Path("/home/bidcoin/df_pdf_v2.csv") 
df_pdf.to_csv(out_path, index=False, encoding="utf-8")

print("저장 완료:", out_path)
print("PDF 행 수:", len(df_pdf))
display(df_pdf.head(20))

저장 완료: /home/bidcoin/df_pdf_v2.csv
PDF 행 수: 6


,공고 번호,공고 차수,사업명,사업 금액,발주 기관,공개 일자,입찰 참여 시작일,입찰 참여 마감일,사업 요약,파일형식,파일명,텍스트,텍스트길이
7,NaN,NaN,차세대 포털·학사 정보시스템 구축사업,1.127000e+10,고려대학교,2024-07-01 00:00:00,2024-07-05 11:00:00,2024-08-12 11:00:00,- 사업개요: 고려대학교 차세대 포털·학사 정보시스템 구축 사업\n- 추진배경: 학...,pdf,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,제 안 요 청 서 \n\n고려대학교 \n차세대 포털·학사 정보시스템 ...,NaN
12,NaN,NaN,[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역,0.000000e+00,서울시립대학교,2023-06-20 00:00:00,NaN,NaN,- 사업개요: 학업성취도 다차원 종단분석 통합시스템 1차 고도화\n- 추진배경: 서...,pdf,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,제안요청서 본 제안요청서는 입찰참여의 균등한 기회 제공을 위해 규격을 공개하기 위한...,102589.0
25,R25BK00601569,1.0,아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아)사업 PMC 용역,9.772400e+08,한국농어촌공사,2025-02-05 18:31:37,2025-02-06 10:00:00,2025-03-10 10:00:00,- 사업개요: 2025년까지 아세안+3 식량안보정보시스템 3단계 협력사업을 캄보디아...,pdf,한국농어촌공사_아세안+3+식량안보정보시스템(AFSIS)+3단계+협력(캄보디아.hwp...,\r\n \r\n 아세안+3 식량안보정보시스템 3단계 협력사업(캄보디아) \...,NaN
39,20240404154,0.0,2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용역,4.937630e+08,서울특별시,2024-04-02 15:49:39,2024-04-19 09:00:00,2024-04-23 16:00:00,- 사업개요: 2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 구축\n-...,pdf,서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf,제 안 요 청 서\n\n사 업 명\n\n주관기관\n\n2024년 지도정...,NaN
42,20241139040,0.0,대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전산시스템 구축,6.000000e+07,대전대학교,2024-11-27 11:36:47,NaN,2024-12-09 11:00:00,- 사업개요: 대전대학교에서 다층적 융합 학습경험 플랫폼(MILE)구축 사업을 추진...,pdf,대전대학교_대전대학교+2024학년도+다층적+융합+학습경험+플랫폼(MILE)+전.hw...,\r\n \r\n(재공고)대전대학교 다층적 융합 학습경험\r\n플랫폼(MI...,NaN
50,20241120435,0.0,2025년도 중이온가속기용 극저온시스템 운전 용역,7.430700e+08,기초과학연구원,2024-11-15 09:53:46,2024-11-22 10:00:00,2024-11-26 14:00:00,- 사업 개요: 2025년도 중이온가속기용 극저온시스템 운전 용역\n- 추진배경: ...,pdf,기초과학연구원_2025년도 중이온가속기용 극저온시스템 운전 용역.pdf,2025년도 중이온가속기용 극저온시스템\n\n운전 용역 과업지시서\n\n20...,NaN


### (1-2)hwp 재파싱 및 노이즈제거

In [264]:
FILES_DIR = "/home/shared/files"

def get_hwp_text_final_exorcism(file_path):
    def is_clean_hangul(char):
        if not '가' <= char <= '힣':
            return True
        try:
            code = char.encode('cp949')
            return 0xB0 <= code[0] <= 0xC8
        except:
            return False

    try:
        f = olefile.OleFileIO(file_path)
        dirs = f.listdir()
        bodytext_sections = [d for d in dirs if 'BodyText' in d]

        raw_text = ""
        for section in bodytext_sections:
            data = f.openstream(section).read()
            decompressed = zlib.decompress(data, -15)
            raw_text += decompressed.decode('utf-16', errors='ignore')

        # 이미지 정보 및 불필요한 메타데이터 제거
        text = re.sub(r'원본 그림의 이름:.*?pixel', ' ', raw_text, flags=re.DOTALL)
        text = re.sub(r'가로 \d+pixel, 세로 \d+pixel', ' ', text)
        text = re.sub(r'[^가-힣a-zA-Z0-9\s\.\(\)\[\]\/\,\%\:\-\·\?\!]', ' ', text)

        tokens = text.split()
        clean_tokens = []

        for t in tokens:
            if all(is_clean_hangul(c) for c in t):
                if len(t) == 1 and t not in '이가을를에와과도한1234567890o-·':
                    continue
                if re.search(r'[a-zA-Z][가-힣]', t) or re.search(r'[가-힣][a-zA-Z]', t):
                    continue
                clean_tokens.append(t)

        return " ".join(clean_tokens)

    except Exception as e:
        print(f"추출 실패: {file_path} / {e}")
        return None


remaining_files = [
    "한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp",
    "재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp",
    "2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp",
    "전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp",
    "한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp",
    "국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp"
]

print("HWP 복구를 시작합니다...")
print("-" * 40)

for file_name in remaining_files:
    file_path = os.path.join(FILES_DIR, file_name)
    idx_list = df[df["파일명"] == file_name].index

    if not idx_list.empty:
        idx = idx_list[0]
        print(f"처리 중: {file_name}")

        restored_text = get_hwp_text_final_exorcism(file_path)

        if restored_text and len(restored_text) > 100:
            df.loc[idx, "텍스트"] = restored_text
            df.loc[idx, "텍스트길이"] = len(restored_text)
            print(f"복구 성공! (최종 길이: {len(restored_text):,}자)")
        else:
            print("텍스트가 너무 짧거나 추출 실패")
    else:
        print(f"df에서 파일명을 찾지 못함: {file_name}")

print("-" * 40)
print("HWP 복구 완료")

display(df[df["파일명"].isin(remaining_files)][["파일명", "텍스트길이"]])

HWP 복구를 시작합니다...
----------------------------------------
처리 중: 한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp
복구 성공! (최종 길이: 54,439자)
처리 중: 재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp
복구 성공! (최종 길이: 35,327자)
처리 중: 2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp
복구 성공! (최종 길이: 26,973자)
처리 중: 전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp
복구 성공! (최종 길이: 34,048자)
처리 중: 한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp
복구 성공! (최종 길이: 27,835자)
처리 중: 국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp
복구 성공! (최종 길이: 31,210자)
----------------------------------------
HWP 복구 완료


,파일명,텍스트길이
2,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,54439.0
8,재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp,35327.0
17,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp,26973.0
18,한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp,27835.0
20,전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp,34048.0
49,국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp,31210.0


In [265]:
check_files = [
    "한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp",
    "재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp",
    "2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp",
    "전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp",
    "한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp",
    "국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp"
]

for file_name in check_files:
    row = df.loc[df["파일명"] == file_name, ["파일명", "텍스트"]]
    print("파일명:", file_name)

    if row.empty:
        print("df에서 해당 파일을 찾지 못했습니다.")
        continue

    text = row["텍스트"].fillna("").iloc[0]
    print("텍스트길이:", len(text))
    print()

파일명: 한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp
텍스트길이: 54439

파일명: 재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp
텍스트길이: 35327

파일명: 2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp
텍스트길이: 26973

파일명: 전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp
텍스트길이: 34048

파일명: 한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp
텍스트길이: 27835

파일명: 국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp
텍스트길이: 31210



In [266]:
short = df.loc[df["텍스트"].fillna("").str.len() <= 500, ["파일명", "텍스트"]].copy()
short["텍스트길이"] = short["텍스트"].fillna("").str.len()

print(short[["파일명", "텍스트길이"]])

Empty DataFrame
Columns: [파일명, 텍스트길이]
Index: []


#### 텍스트 살펴보기 - hwp

In [267]:
df['발주 기관'].unique()

<ArrowStringArray>
[                      '한영대학',                     '한국연구재단',
                  '한국생산기술연구원',                      '인천광역시',
                   '경상북도 봉화군',                   '한국전기안전공사',
                  '재단법인충북연구원',                      '고려대학교',
                '재단법인스포츠윤리센터',                    '국방과학연구소',
              '(사）한국대학스포츠협의회',                   '한국사학진흥재단',
                    '서울시립대학교',                      '경희대학교',
                    '한국수자원공사',              '국가과학기술지식정보서비스',
                '한국철도공사 (용역)', '2025 구미 아시아육상경기선수권대회 조직위원회',
               '한국발명진흥회 입찰공고',                   '고양도시관리공사',
                      '전북대학교',                  '한국보건산업진흥원',
                  '한국사회보장정보원',                      '수협중앙회',
                    '한국농어촌공사',                 'KOICA 전자조달',
                   '대한장애인체육회',                   '인천광역시 동구',
                   '축산물품질평가원',                      '울산광역시',
                    '한국재정정보원',                     '부산관광공사',
     

In [268]:
df['발주 기관'].nunique()

87

In [269]:
len(df['발주 기관'])

100

#### 기관섹터 매핑

In [270]:
# 기관유형별로 문서를 묶어보자는 기준을 세워 섹터 나눠보고자 함

In [271]:
# 조직형태 기준으로 섹터 7개

# "중앙행정기관": central
# "지자체": local
# "공기업/공공기관": public
# "대학/교육기관": education
# "연구기관": research
# "재단/협회/비영리": nonprofit
# "민간기업": company

In [272]:
# 발주기관 unique값들 각 섹터로 매핑
def get_org_sector_map():
    central = [
        "대검찰청",
        "중앙선거관리위원회",
    ]

    local = [
        "서울특별시교육청",
        "경기도 안양시",
        "경기도 평택시",
        "경상북도 봉화군",
        "인천광역시 동구",
        "서울특별시",
        "울산광역시",
        "인천광역시",
        "전북특별자치도 정읍시",
    ]

    public = [
        "국립인천해양박물관",
        "국가과학기술지식정보서비스",
        "BioIN",
        "재단법인경기도일자리재단",
        "서울특별시 여성가족재단",
        "재단법인 광주광역시 광주문화재단",
        "경기도사회서비스원",
        "세종테크노파크",
        "대한장애인체육회",
        "한국연구재단",
        "한국사학진흥재단",
        "한국보건산업진흥원",
        "한국사회보장정보원",
        "한국재정정보원",
        "한국로봇산업진흥원",
        "한국건강가정진흥원",
        "한국보육진흥원",
        "서민금융진흥원",
        "한국발명진흥회 입찰공고",
        "한국지식재산보호원",
        "축산물품질평가원",
        "한국교육과정평가원",
        "국립중앙의료원",
        "국가철도공단",
        "국민연금공단",
        "한국산업인력공단",
        "한국어촌어항공단",
        "한국산업단지공단",
        "한국전기안전공사",
        "한국수자원공사",
        "한국철도공사 (용역)",
        "고양도시관리공사",
        "부산관광공사",
        "파주도시관광공사",
        "한국농어촌공사",
        "한국농수산식품유통공사",
        "한국가스공사",
        "그랜드코리아레저(주)",
        "인천공항운영서비스(주)",
        "한국수출입은행",
        "KOICA 전자조달",
        "재단법인스포츠윤리센터",
        "한국해양조사협회",
        "대한적십자사 의료원",
        "(재)예술경영지원센터",
        "재단법인 한국장애인문화예술원",
        "문화체육관광부 국립민속박물관",
    ]

    education = [
        "경희대학교",
        "고려대학교",
        "광주과학기술원",
        "남서울대학교",
        "대전대학교",
        "서영대학교 산학협력단",
        "서울시립대학교",
        "을지대학교",
        "전북대학교",
        "조선대학교",
        "한영대학",
    ]

    research = [
        "국방과학연구소",
        "재단법인충북연구원",
        "재단법인 광주연구원",
        "한국생산기술연구원",
        "기초과학연구원",
        "한국한의학연구원",
        "한국원자력연구원",
        "나노종합기술원",
        "한국수자원조사기술원",
    ]

    nonprofit = [
        "대한상공회의소",
        "사단법인아시아물위원회사무국",
        "사단법인 보험개발원",
        "(사)벤처기업협회",
        "(사)부산국제영화제",
        "(사）한국대학스포츠협의회",
        "2025 구미 아시아육상경기선수권대회 조직위원회",
        "수협중앙회",
    ]

    company = [
        "케빈랩 주식회사",
    ]

    sector_groups = {
        "중앙행정기관(central)": central,
        "지자체(local)": local,
        "공기업/공공기관(public)": public,
        "대학/교육기관(education)": education,
        "연구기관(research)": research,
        "재단/협회/비영리(nonprofit)": nonprofit,
        "민간기업(company)": company,
    }

    org_sector_map = {
        org: sector
        for sector, org_list in sector_groups.items()
        for org in org_list
    }

    return org_sector_map

In [273]:
# 기관 매핑
df["기관 섹터"] = df["발주 기관"].map(get_org_sector_map())

In [274]:
print("기관섹터 null 개수:", df["기관 섹터"].isna().sum())

기관섹터 null 개수: 0


In [275]:
df["기관 섹터"].value_counts().sum()

np.int64(100)

In [276]:
# 매핑 검증
is_hwp = df["파일명"].str.lower().str.endswith(".hwp")

df.loc[is_hwp, ["기관 섹터", "발주 기관"]] \
  .drop_duplicates() \
  .sort_values(["기관 섹터", "발주 기관"]) \
  .groupby("기관 섹터")["발주 기관"] \
  .apply(list)

sector_orgs = (
    df[["기관 섹터", "발주 기관"]]
      .dropna(subset=["기관 섹터", "발주 기관"])
      .drop_duplicates()
      .sort_values(["기관 섹터", "발주 기관"])
      .groupby("기관 섹터")["발주 기관"]
      .apply(list)
)

for sector, orgs in sector_orgs.items():
    print(f"■ {sector} (총 {len(orgs)}개)")
    for org in orgs:
        print(f"  - {org}")
    print("-" * 50)


■ 공기업/공공기관(public) (총 47개)
  - (재)예술경영지원센터
  - BioIN
  - KOICA 전자조달
  - 경기도사회서비스원
  - 고양도시관리공사
  - 국가과학기술지식정보서비스
  - 국가철도공단
  - 국립인천해양박물관
  - 국립중앙의료원
  - 국민연금공단
  - 그랜드코리아레저(주)
  - 대한장애인체육회
  - 대한적십자사 의료원
  - 문화체육관광부 국립민속박물관
  - 부산관광공사
  - 서민금융진흥원
  - 서울특별시 여성가족재단
  - 세종테크노파크
  - 인천공항운영서비스(주)
  - 재단법인 광주광역시 광주문화재단
  - 재단법인 한국장애인문화예술원
  - 재단법인경기도일자리재단
  - 재단법인스포츠윤리센터
  - 축산물품질평가원
  - 파주도시관광공사
  - 한국가스공사
  - 한국건강가정진흥원
  - 한국교육과정평가원
  - 한국농수산식품유통공사
  - 한국농어촌공사
  - 한국로봇산업진흥원
  - 한국발명진흥회 입찰공고
  - 한국보건산업진흥원
  - 한국보육진흥원
  - 한국사학진흥재단
  - 한국사회보장정보원
  - 한국산업단지공단
  - 한국산업인력공단
  - 한국수자원공사
  - 한국수출입은행
  - 한국어촌어항공단
  - 한국연구재단
  - 한국재정정보원
  - 한국전기안전공사
  - 한국지식재산보호원
  - 한국철도공사 (용역)
  - 한국해양조사협회
--------------------------------------------------
■ 대학/교육기관(education) (총 11개)
  - 경희대학교
  - 고려대학교
  - 광주과학기술원
  - 남서울대학교
  - 대전대학교
  - 서영대학교 산학협력단
  - 서울시립대학교
  - 을지대학교
  - 전북대학교
  - 조선대학교
  - 한영대학
--------------------------------------------------
■ 민간기업(company) (총 1개)
  - 케빈랩 주식회사
-------------------------

In [277]:
df.loc[is_hwp, "기관 섹터"].value_counts()  # 96개

기관 섹터
공기업/공공기관(public)        54
연구기관(research)          10
대학/교육기관(education)       9
지자체(local)               9
재단/협회/비영리(nonprofit)     9
중앙행정기관(central)          2
민간기업(company)            1
Name: count, dtype: int64

In [278]:
df.columns

Index(['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일',
       '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '텍스트', '텍스트길이', '기관 섹터'],
      dtype='str')

In [279]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   공고 번호      82 non-null     str    
 1   공고 차수      82 non-null     float64
 2   사업명        100 non-null    str    
 3   사업 금액      99 non-null     float64
 4   발주 기관      100 non-null    str    
 5   공개 일자      100 non-null    str    
 6   입찰 참여 시작일  74 non-null     str    
 7   입찰 참여 마감일  92 non-null     str    
 8   사업 요약      100 non-null    str    
 9   파일형식       100 non-null    str    
 10  파일명        100 non-null    str    
 11  텍스트        100 non-null    str    
 12  텍스트길이      7 non-null      float64
 13  기관 섹터      100 non-null    str    
dtypes: float64(3), str(11)
memory usage: 1.5 MB


#### 텍스트 공통정제

공통 정제 함수로 전체적으로 거른 뒤 섹터별 함수로 세부적 정제

In [280]:
# 공통 정제 함수
def clean_text_common(text):
    """
    최소 전처리 공통 함수
    - 의미/구조를 바꿀 수 있는 공격적 정제는 하지 않음
    - 모든 섹터에 공통으로 안전한 정리만 수행
    """
    if pd.isna(text):
        return text

    text = str(text)

    # 1) 줄바꿈 통일
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # 2) 탭 정리
    text = text.replace("\t", " ")

    # 3) 명백히 깨진 일부 특수문자 제거
    text = re.sub(r"[↸ᬄὩ⇟]", " ", text)

    # 4) 제어문자 제거
    text = re.sub(r"[\x00-\x08\x0b-\x1f\x7f]", " ", text)

    # 5) 줄 내부의 연속 공백만 축소
    #    줄바꿈 구조는 건드리지 않음
    text = re.sub(r"[ ]{2,}", " ", text)

    # 6) 줄 양끝 공백 제거
    text = "\n".join(line.strip() for line in text.split("\n"))

    # 7) 과도한 빈 줄만 축소
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [281]:
df["공통정제텍스트"] = df["텍스트"].apply(clean_text_common)

In [282]:
len(df["공통정제텍스트"])

100

In [283]:
# 섹터별 정제함수

# clean_text_common적용한 공통정제텍스트와 동일한 정제테스트 컬럼만들고,
# 정제테스트 컬럼에 기관섹터별 함수(ex.clean_text_public) 적용한 텍스트를 덮어씀,
# 공통정제텍스트 컬럼은 그대로 보존 //
# 정제텍스트 컬럼은 처음엔 공통정제텍스트 복사본으로 시작하는 것,
# 그 다음 public, rnd, local ... 함수가 해당 행만 덮어쓰면서 누적 반영 //

df["정제텍스트"] = df["공통정제텍스트"].copy()

#### 1. 공기업/공공기관(public)

In [284]:
def clean_text_public(text):
    """
    공기업/공공기관(public) 전용 보수적 정제

    원칙
    - 연락처/이메일/URL은 보호
    - 제목형 띄어쓰기만 제한적으로 정리
    - 2줄짜리 짧은 표제/장제목 일부 결합
    - 목차형 줄 끝 페이지번호 제거
    - 확실한 OCR/이미지 잔여물만 제거
    - 본문 의미 훼손 가능성이 있는 과도한 치환은 지양
    """

    if pd.isna(text):
        return text

    raw = str(text).strip()
    if not raw:
        return raw

    # --------------------------------------------------
    # 0) 입력 전체가 보호 대상이면 그대로 반환
    # --------------------------------------------------
    full_protected_patterns = [
        r"^\d{2,4}-\d{2,4}-\d{3,4}$",                         # 063-716-2787
        r"^(?:TEL|FAX)\s*:\s*\d{2,4}-\d{2,4}-\d{3,4}$",       # TEL: 042-615-5670
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",  # email
        r"^https?://\S+$",                                    # url
    ]
    for pat in full_protected_patterns:
        if re.fullmatch(pat, raw, flags=re.IGNORECASE):
            return raw

    # --------------------------------------------------
    # 1) 공통 정제
    # --------------------------------------------------
    text = clean_text_common(raw)

    # 자주 보이는 표기 통일
    text = text.replace("㈜", "(주)")
    text = text.replace("（", "(").replace("）", ")")

    # --------------------------------------------------
    # 2) 줄 단위 보호 패턴
    # --------------------------------------------------
    protected_line_pattern = re.compile(
        r"("
        r"^\d{2,4}-\d{2,4}-\d{3,4}$"
        r"|(?:TEL|FAX)\s*:\s*\d{2,4}-\d{2,4}-\d{3,4}"
        r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
        r"|https?://\S+"
        r")",
        re.IGNORECASE
    )

    # --------------------------------------------------
    # 3) 2줄짜리 짧은 표제 결합
    # --------------------------------------------------
    merge_pairs = {
        ("목", "차"): "목차",
        ("순", "서"): "순서",
        ("담", "당"): "담당",
        ("소", "속"): "소속",
        ("성", "명"): "성명",
        ("직", "위"): "직위",
        ("전", "화"): "전화",
        ("부", "서"): "부서",
        ("부서", "명"): "부서명",
        ("전화", "번호"): "전화번호",
        ("주", "관"): "주관",
        ("기", "관"): "기관",
        ("주관", "기관"): "주관기관",
        ("사", "업"): "사업",
        ("개", "요"): "개요",
        ("붙", "임"): "붙임",
        ("별", "표"): "별표",
        ("별", "지"): "별지",
        ("서", "식"): "서식",
    }

    roman_only_pattern = re.compile(r"^[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+[\.．]?$")

    lines = [line.strip() for line in text.split("\n")]
    merged_lines = []
    i = 0

    while i < len(lines):
        cur = lines[i]

        # 빈 줄은 그대로
        if not cur:
            merged_lines.append("")
            i += 1
            continue

        if i + 1 < len(lines):
            nxt = lines[i + 1].strip()

            # 3-1) 짧은 2줄 표제 결합
            merged = merge_pairs.get((cur, nxt))
            if merged:
                merged_lines.append(merged)
                i += 2
                continue

            # 3-2) 로마숫자 단독 줄 + 다음 줄 제목 결합
            # 예: "Ⅰ" + "사업 개요" -> "Ⅰ. 사업 개요"
            if roman_only_pattern.fullmatch(cur) and nxt:
                merged_lines.append(cur.rstrip(".．") + ". " + nxt)
                i += 2
                continue

        merged_lines.append(cur)
        i += 1

    # --------------------------------------------------
    # 4) 보수적으로 붙여도 되는 짧은 표제 사전
    # --------------------------------------------------
    header_compact_map = {
        "제 안 요 청 서": "제안요청서",
        "과 업 지 시 서": "과업지시서",
        "사 업 명": "사업명",
        "사 업 비": "사업비",
        "사 업 개 요": "사업개요",
        "사 업 범 위": "사업범위",
        "사 업 기 간": "사업기간",
        "추 진 배 경": "추진배경",
        "추 진 방 안": "추진방안",
        "추 진 목 표": "추진목표",
        "추 진 일 정": "추진일정",
        "주 관 기 관": "주관기관",
        "수 요 기 관": "수요기관",
        "부 서 명": "부서명",
        "전 화 번 호": "전화번호",
        "담 당 자": "담당자",
        "목 차": "목차",
        "순 서": "순서",
        "일 반 사 항": "일반사항",
        "기 대 효 과": "기대효과",
        "제 안 안 내 사 항": "제안 안내사항",
        "제 안 서 작 성 요 령": "제안서 작성요령",
        "제 안 요 청 내 용": "제안요청 내용",
        "정 보 시 스 템 현 황": "정보시스템 현황",
    }

    # --------------------------------------------------
    # 5) 목차형 패턴
    # --------------------------------------------------
    toc_head_pattern = re.compile(
        r"""^(
            \[.*\]|
            <.*>|
            [ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+[\.．]?\s*|
            제?\d+장\s+|
            \d+(\.\d+)*[\.\)]\s+|
            [가나다라마바사아자차카타파하][\.\)]\s+|
            (별지|별표|붙임|서식|첨부서식)\s*\d*\.?\s*
        )""",
        re.VERBOSE
    )

    roman_prefix_pattern = re.compile(r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+[\.．]?\s*)(.+)$")

    # 번호 항목 시작부에서만 보수적으로 붙일 제목들
    numbered_header_terms = [
        "사 업 명",
        "사 업 비",
        "사 업 기 간",
        "사 업 범 위",
        "사 업 개 요",
        "추 진 배 경",
        "추 진 방 안",
        "추 진 목 표",
        "추 진 일 정",
        "사 업 내 용",
        "사 업 예 산",
        "기 대 효 과",
    ]
    numbered_header_pattern = re.compile(
        r"^(\d+\.\s*)(" + "|".join(re.escape(x) for x in numbered_header_terms) + r")(\s*[:：]?\s*)(.*)$"
    )
    new_lines = []

    for line in merged_lines:
        s = line.strip()

        if not s:
            new_lines.append("")
            continue

        # 보호 라인 그대로 유지
        if protected_line_pattern.search(s):
            new_lines.append(s)
            continue

        # ----------------------------------------------
        # 5-1) 확실한 OCR/이미지 잔여물 제거
        # ----------------------------------------------
        if re.fullmatch(r"(세로|가로)\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue

        # 기호만 있는 짧은 줄 제거
        if re.fullmatch(r"[0Oo○◦·•\-=]{1,4}", s):
            continue

        # 숫자만 있는 아주 짧은 줄 제거 (고립 페이지번호 가능성)
        if re.fullmatch(r"\d{1,2}", s):
            continue

        # ----------------------------------------------
        # 5-2) 표제 사전 치환
        # ----------------------------------------------
        if s in header_compact_map:
            s = header_compact_map[s]

        # ----------------------------------------------
        # 5-3) 번호 항목 시작부의 제목형 띄어쓰기 정리
        # 예: "1. 사 업 명 :" -> "1. 사업명 :"
        # ----------------------------------------------
        s = numbered_header_pattern.sub(
            lambda m: m.group(1) + m.group(2).replace(" ", "") + m.group(3) + m.group(4),
            s
        )

        # ----------------------------------------------
        # 5-4) 짧은 제목형 띄어쓰기 정리
        # 너무 일반적인 본문에는 적용하지 않음
        # ----------------------------------------------
        if (
            len(s) <= 20
            and ":" not in s
            and not re.search(r"\d{2,}", s)
            and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,15}[가-힣A-Za-z]", s)
        ):
            s = s.replace(" ", "")

        # ----------------------------------------------
        # 5-5) 로마숫자 장/절 제목 정리
        # 예: "Ⅰ. 사 업 개 요" -> "Ⅰ. 사업 개요"
        # ----------------------------------------------
        m = roman_prefix_pattern.match(s)
        if m:
            prefix, body = m.group(1), m.group(2).strip()

            if (
                len(body) <= 30
                and not re.search(r"\d{2,}", body)
                and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,20}[가-힣A-Za-z]", body)
            ):
                compact = body.replace(" ", "")
                spaced_title_map = {
                    "사업개요": "사업 개요",
                    "사업추진방안": "사업추진 방안",
                    "정보시스템현황": "정보시스템 현황",
                    "제안요청내용": "제안요청 내용",
                    "제안안내사항": "제안 안내사항",
                    "제안서작성요령": "제안서 작성요령",
                    "운영환경": "운영 환경",
                }
                body = spaced_title_map.get(compact, compact)
                s = prefix + body

        # ----------------------------------------------
        # 5-6) 괄호 안 제목형 띄어쓰기
        # ----------------------------------------------
        def fix_spaced_korean_in_parens(m):
            inner = m.group(1)
            if (
                len(inner) <= 20
                and not re.search(r"\d{2,}", inner)
                and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,15}[가-힣A-Za-z]", inner)
            ):
                return f"({inner.replace(' ', '')})"
            return m.group(0)

        s = re.sub(
            r"\(((?:[가-힣A-Za-z]\s){1,15}[가-힣A-Za-z])\)",
            fix_spaced_korean_in_parens,
            s
        )

        # ----------------------------------------------
        # 5-7) 목차형 줄 끝 페이지번호 제거
        # 예: "- 7", "- - 62", "··· 12"
        # ----------------------------------------------
        if toc_head_pattern.match(s):
            s = re.sub(r"\s*-\s*-?\s*\d{1,3}(?:\s*[xX])?\s*$", "", s)
            s = re.sub(r"\s*[·•‧\.\…]{2,}\s*\d{1,3}\s*$", "", s)

        # 목차 아닌 줄에서도 아주 전형적인 짧은 제목줄의 끝 페이지번호만 제거
        if (
            len(s) <= 60
            and ":" not in s
            and re.search(r"\s-\s\d{1,3}$", s)
        ):
            s = re.sub(r"\s-\s\d{1,3}$", "", s)

        new_lines.append(s)

    text = "\n".join(new_lines)

    # --------------------------------------------------
    # 6) 후처리
    # --------------------------------------------------
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

#### 2. 연구기관(research)

In [285]:
def clean_text_research(text):
    """
    연구기관(research) 전용 보수적 정제

    원칙
    - 연락처/이메일/URL은 보호
    - 제목형 띄어쓰기만 제한적으로 정리
    - 2줄짜리 짧은 표제/장제목 일부 결합
    - 목차형 줄 끝 페이지번호 제거
    - 확실한 OCR/이미지 잔여물만 제거
    - 본문 의미 훼손 가능성이 있는 과도한 치환은 지양
    - 표지/기관명/사업명 같은 핵심 메타정보는 보존
    """

    if pd.isna(text):
        return text

    raw = str(text).strip()
    if not raw:
        return raw

    # --------------------------------------------------
    # 0) 입력 전체가 보호 대상이면 그대로 반환
    # --------------------------------------------------
    full_protected_patterns = [
        r"^\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^(?:TEL|FAX)\s*:\s*\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
        r"^https?://\S+$",
    ]
    for pat in full_protected_patterns:
        if re.fullmatch(pat, raw, flags=re.IGNORECASE):
            return raw

    # --------------------------------------------------
    # 1) 공통 정제
    # --------------------------------------------------
    text = clean_text_common(raw)

    text = text.replace("㈜", "(주)")
    text = text.replace("（", "(").replace("）", ")")

    # --------------------------------------------------
    # 2) 문서 앞부분 OCR 깨짐 헤더 제한 제거
    #    너무 공격적으로 지우지 않도록 약하게 조정
    #    - 첫 줄이 매우 지저분할 때만
    #    - 제안요청서/사업명/과업명/기관명이 바로 뒤에 나오는 경우만
    # --------------------------------------------------
    front_noise_pattern = re.compile(
        r"^([^\n]{0,120})\n",
        re.MULTILINE
    )
    m = front_noise_pattern.match(text)
    if m:
        first_line = m.group(1).strip()
        rest = text[m.end():]

        looks_noisy = (
            len(first_line) >= 12
            and (
                len(re.findall(r"[가-힣A-Za-z0-9]", first_line)) / max(len(first_line), 1) < 0.7
                or len(re.findall(r"\b[가-힣A-Za-z0-9]\b", first_line)) >= 6
                or bool(re.search(r"[쌀뀀툀턀럀]+", first_line))
            )
        )

        next_has_anchor = bool(re.search(
            r"(제\s*안\s*요\s*청\s*서|제안요청서|사업명|과업명|발주기관|주관기관|20\d{2}\.\s*\d{1,2}\.)",
            rest[:300]
        ))

        if looks_noisy and next_has_anchor:
            text = rest.lstrip()

    # --------------------------------------------------
    # 3) 보호 줄 패턴
    # --------------------------------------------------
    protected_line_pattern = re.compile(
        r"("
        r"^\d{2,4}-\d{2,4}-\d{3,4}$"
        r"|(?:TEL|FAX)\s*:\s*\d{2,4}-\d{2,4}-\d{3,4}"
        r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
        r"|https?://\S+"
        r")",
        re.IGNORECASE
    )

    # --------------------------------------------------
    # 4) 2줄 제목 결합
    # --------------------------------------------------
    merge_pairs = {
        ("목", "차"): "목차",
        ("순", "서"): "순서",
        ("담", "당"): "담당",
        ("소", "속"): "소속",
        ("성", "명"): "성명",
        ("직", "위"): "직위",
        ("전", "화"): "전화",
        ("부", "서"): "부서",
        ("부서", "명"): "부서명",
        ("전화", "번호"): "전화번호",
        ("사", "업"): "사업",
        ("과", "업"): "과업",
        ("개", "요"): "개요",
        ("현", "황"): "현황",
        ("안", "내"): "안내",
        ("붙", "임"): "붙임",
        ("별", "표"): "별표",
        ("별", "지"): "별지",
        ("서", "식"): "서식",
        ("주", "관"): "주관",
        ("기", "관"): "기관",
        ("주관", "기관"): "주관기관",
        ("발주", "기관"): "발주기관",
    }

    roman_only_pattern = re.compile(
    r"^(?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+|(?=[IVXLC]+$)I|II|III|IV|V|VI|VII|VIII|IX|X|XI|XII|XIII|XIV|XV)[\.．]?$"
    )

    lines = [line.strip() for line in text.split("\n")]
    merged_lines = []
    i = 0

    while i < len(lines):
        cur = lines[i]

        if not cur:
            merged_lines.append("")
            i += 1
            continue

        if i + 1 < len(lines):
            nxt = lines[i + 1].strip()

            merged = merge_pairs.get((cur, nxt))
            if merged:
                merged_lines.append(merged)
                i += 2
                continue

            # 예: "Ⅰ" + "사업 개요" -> "Ⅰ. 사업 개요"
            if roman_only_pattern.fullmatch(cur):
                roman_map = {
                    "I": "Ⅰ", "II": "Ⅱ", "III": "Ⅲ", "IV": "Ⅳ", "V": "Ⅴ",
                    "VI": "Ⅵ", "VII": "Ⅶ", "VIII": "Ⅷ", "IX": "Ⅸ", "X": "Ⅹ"
                }
                cur_norm = roman_map.get(cur.rstrip(".．"), cur.rstrip(".．"))

                # 바로 다음 줄이 비어 있으면 한 줄 더 본다
                if nxt:
                    merged_lines.append(cur_norm + ". " + nxt)
                    i += 2
                    continue
                elif i + 2 < len(lines):
                    nxt2 = lines[i + 2].strip()
                    if nxt2:
                        merged_lines.append(cur_norm + ". " + nxt2)
                        i += 3
                        continue

        merged_lines.append(cur)
        i += 1

    # --------------------------------------------------
    # 5) 표제 사전
    # --------------------------------------------------
    header_compact_map = {
        "제 안 요 청 서": "제안요청서",
        "사 업 명": "사업명",
        "과 업 명": "과업명",
        "발 주 기 관": "발주기관",
        "주 관 기 관": "주관기관",
        "부 서 명": "부서명",
        "전 화 번 호": "전화번호",
        "담 당 자": "담당자",
        "목 차": "목차",
        "< 목 차 >": "<목차>",
        "사 업 개 요": "사업개요",
        "과 업 개 요": "과업개요",
        "사 업 범 위": "사업범위",
        "과 업 범 위": "과업범위",
        "사 업 기 간": "사업기간",
        "과 업 기 간": "과업기간",
        "추 진 배 경": "추진배경",
        "추 진 목 적": "추진목적",
        "추 진 방 안": "추진방안",
        "추 진 계 획": "추진계획",
        "추 진 일 정": "추진일정",
        "기 대 효 과": "기대효과",
        "제 안 요 청 내 용": "제안요청 내용",
        "제 안 안 내 사 항": "제안 안내사항",
        "제 안 서 작 성 요 령": "제안서 작성요령",
        "정 보 시 스 템 현 황": "정보시스템 현황",
        "대 상 업 무 현 황": "대상 업무 현황",
    }

    toc_head_pattern = re.compile(
        r"""^(
            \[.*\]|
            <.*>|
            [ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*|
            제?\d+장\s+|
            \d+(\.\d+)*[\.\)]\s+|
            [가나다라마바사아자차카타파하][\.\)]\s+|
            (별지|별표|붙임|서식|첨부서식|별첨|부록|참고자료)\s*\d*\.?\s*
        )""",
        re.VERBOSE
    )

    roman_prefix_pattern = re.compile(
        r"^((?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+|(?=[IVXLC]+$)[IVXLC]+)[\.．]?\s*)(.+)$"
    )

    # 번호 항목 시작부에서만 붙일 제목
    numbered_header_terms = [
        "사 업 명", "과 업 명",
        "사 업 비", "사 업 예 산",
        "사 업 기 간", "과 업 기 간",
        "사 업 범 위", "과 업 범 위",
        "사 업 개 요", "과 업 개 요",
        "추 진 배 경", "추 진 목 적",
        "추 진 방 안", "추 진 계 획",
        "추 진 목 표", "추 진 일 정",
        "사 업 내 용", "과 업 내 용",
        "기 대 효 과",
    ]
    numbered_header_pattern = re.compile(
        r"^(\d+\.\s*)(" + "|".join(re.escape(x) for x in numbered_header_terms) + r")(\s*[:：]?\s*)(.*)$"
    )

    # 표지 핵심어 보호
    cover_meta_keywords = [
        "제안요청서", "제 안 요 청 서", "제안요청서(RFP)", "RFP",
        "사업명", "과업명", "발주기관", "주관기관",
        "한국생산기술연구원", "국방과학연구소", "충북연구원",
        "광주연구원", "한국원자력연구원", "한국한의학연구원",
        "한국수자원조사기술원"
    ]

    new_lines = []

    for idx, line in enumerate(merged_lines):
        s = line.strip()

        if not s:
            new_lines.append("")
            continue

        # 보호 라인 그대로 유지
        if protected_line_pattern.search(s):
            new_lines.append(s)
            continue

        # 표지 핵심 정보는 삭제/과도치환 방지
        if any(k in s for k in cover_meta_keywords):
            if s in header_compact_map:
                s = header_compact_map[s]
            new_lines.append(s)
            continue

        # ----------------------------------------------
        # 5-1) 확실한 OCR/이미지 잔여물 제거
        # ----------------------------------------------
        if re.fullmatch(r"(세로|가로)\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue

        if re.fullmatch(r"[·•○◦\-_=~]{1,6}", s):
            continue

        # 숫자 단독 줄 제거는 더 보수적으로:
        # 문서 첫머리/표/절번호 손상 막기 위해 1자리만 제거
        if re.fullmatch(r"\d", s):
            continue

        # ----------------------------------------------
        # 5-2) 이상한 목차 노이즈 문자 보정
        # 예: "1. h 사업 개요" -> "1. 사업 개요"
        # ----------------------------------------------
        s = re.sub(r"^(\d+\.\s*)[hH]\s+", r"\1", s)

        # ----------------------------------------------
        # 5-3) 표제 사전 치환
        # ----------------------------------------------
        if s in header_compact_map:
            s = header_compact_map[s]

        # ----------------------------------------------
        # 5-4) 번호 항목 시작부 제목형 정리
        # 예: "1. 사 업 명 : ..." -> "1. 사업명 : ..."
        # ----------------------------------------------
        s = numbered_header_pattern.sub(
            lambda m: m.group(1) + m.group(2).replace(" ", "") + m.group(3) + m.group(4),
            s
        )

        # ----------------------------------------------
        # 5-5) 짧은 제목형 띄어쓰기 정리
        # ----------------------------------------------
        if (
            len(s) <= 35
            and ":" not in s
            and not re.search(r"\d{3,}", s)
            and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,25}[가-힣A-Za-z]", s)
        ):
            s = s.replace(" ", "")

        # ----------------------------------------------
        # 5-6) 로마숫자 장/절 제목 정리
        # 예: "Ⅰ. 사 업 개 요" -> "Ⅰ. 사업 개요"
        # ----------------------------------------------
        m = roman_prefix_pattern.match(s)
        if m:
            prefix, body = m.group(1), m.group(2).strip()

            if (
                len(body) <= 35
                and not re.search(r"\d{3,}", body)
                and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,25}[가-힣A-Za-z]", body)
            ):
                compact = body.replace(" ", "")
                spaced_title_map = {
                    "사업개요": "사업 개요",
                    "과업개요": "과업 개요",
                    "사업추진방안": "사업 추진방안",
                    "사업추진계획": "사업 추진 계획",
                    "정보시스템현황": "정보시스템 현황",
                    "대상업무현황": "대상 업무 현황",
                    "제안요청내용": "제안요청 내용",
                    "제안안내사항": "제안 안내사항",
                    "제안서작성요령": "제안서 작성요령",
                }
                body = spaced_title_map.get(compact, compact)
                s = prefix + body

        # ----------------------------------------------
        # 5-7) 괄호 안 제목형 띄어쓰기
        # ----------------------------------------------
        def fix_spaced_korean_in_parens(m):
            inner = m.group(1)
            if (
                len(inner) <= 35
                and not re.search(r"\d{3,}", inner)
                and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,25}[가-힣A-Za-z]", inner)
            ):
                return f"({inner.replace(' ', '')})"
            return m.group(0)

        s = re.sub(
            r"\(((?:[가-힣A-Za-z]\s){1,25}[가-힣A-Za-z])\)",
            fix_spaced_korean_in_parens,
            s
        )

        # ----------------------------------------------
        # 5-8) 목차형 줄 끝 페이지번호 제거
        # ----------------------------------------------
        if toc_head_pattern.match(s):
            s = re.sub(r"\s*-\s*-?\s*\d{1,3}(?:\s*[xX])?\s*$", "", s)
            s = re.sub(r"\s*[·•‧\.\…]{2,}\s*\d{1,3}\s*$", "", s)

        # 짧은 제목줄 끝 페이지번호 제거
        if (
            len(s) <= 70
            and ":" not in s
            and re.search(r"\s-\s\d{1,3}$", s)
        ):
            s = re.sub(r"\s-\s\d{1,3}$", "", s)

        new_lines.append(s)

    text = "\n".join(new_lines)

    # --------------------------------------------------
    # 6) 후처리
    # --------------------------------------------------
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

#### 3. 대학/교육기관(education)

In [286]:
def clean_text_education(text):
    """
    대학/교육기관(education) 전용 보수적 정제

    원칙
    - 연락처/이메일/URL은 보호
    - 제목형 띄어쓰기만 제한적으로 정리
    - 장/절 제목이 줄바꿈으로 분리된 경우 일부 결합
    - 목차형 줄 끝 페이지번호 제거
    - 확실한 OCR/이미지 잔여물만 제거
    - 본문/표 의미 훼손 가능성이 있는 과도한 삭제는 지양
    """
    if pd.isna(text):
        return text

    raw = str(text).strip()
    if not raw:
        return raw

    # --------------------------------------------------
    # 0) 입력 전체가 보호 대상이면 그대로 반환
    # --------------------------------------------------
    full_protected_patterns = [
        r"^\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^(?:TEL|FAX|Tel\.?|Fax\.?|전화)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
        r"^https?://\S+$",
    ]
    for pat in full_protected_patterns:
        if re.fullmatch(pat, raw, flags=re.IGNORECASE):
            return raw

    # --------------------------------------------------
    # 1) 공통 정제
    # --------------------------------------------------
    text = clean_text_common(raw)
    text = text.replace("㈜", "(주)")
    text = text.replace("（", "(").replace("）", ")")
    text = re.sub(r"[–—−]", "-", text)

    # --------------------------------------------------
    # 2) 보호 줄 패턴
    # --------------------------------------------------
    protected_line_pattern = re.compile(
        r"("
        r"^\d{2,4}-\d{2,4}-\d{3,4}$"
        r"|(?:TEL|FAX|Tel\.?|Fax\.?|전화)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}"
        r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
        r"|https?://\S+"
        r")",
        re.IGNORECASE
    )

    # --------------------------------------------------
    # 3) 줄 단위 제목 결합
    # --------------------------------------------------
    lines = [line.strip() for line in text.split("\n")]
    merged_lines = []
    i = 0

    merge_pairs = {
        ("목", "차"): "목차",
        ("개", "요"): "개요",
        ("현", "황"): "현황",
        ("붙", "임"): "붙임",
        ("별", "표"): "별표",
        ("별", "지"): "별지",
        ("서", "식"): "서식",
        ("안", "내"): "안내",
        ("사", "업"): "사업",
        ("제안요청", "사항"): "제안요청 사항",
        ("제안안내", "사항"): "제안안내 사항",
    }

    roman_only_pattern = re.compile(
        r"^(?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+|[IVXLC]+)[\.．]?$"
    )

    # 로마숫자 뒤에 붙이기 좋은 교육기관형 장 제목
    edu_section_title_pattern = re.compile(
        r"(?:[가-힣A-Za-z]\s){0,25}[가-힣A-Za-z]"
    )

    spaced_title_map = {
        "사업안내": "사업 안내",
        "사업개요": "사업 개요",
        "구축방안": "구축 방안",
        "제안요청내용": "제안요청 내용",
        "제안요청사항": "제안요청 사항",
        "제안안내사항": "제안안내 사항",
        "참조자료": "참조자료",
        "사업추진방안": "사업 추진방안",
        "사업추진방향": "사업 추진방향",
        "사업배경및목적": "사업배경 및 목적",
    }

    while i < len(lines):
        cur = lines[i]

        if not cur:
            merged_lines.append("")
            i += 1
            continue

        if i + 1 < len(lines):
            nxt = lines[i + 1].strip()

            # 보호 줄은 건드리지 않음
            if protected_line_pattern.search(cur) or protected_line_pattern.search(nxt):
                merged_lines.append(cur)
                i += 1
                continue

            # 1) 미리 정의한 짧은 표제 결합
            merged = merge_pairs.get((cur, nxt))
            if merged:
                merged_lines.append(merged)
                i += 2
                continue

            # 2) 로마숫자 장 제목: Ⅰ / 사업 안내 -> Ⅰ. 사업 안내
            if roman_only_pattern.fullmatch(cur):
                roman_map = {
                    "I": "Ⅰ", "II": "Ⅱ", "III": "Ⅲ", "IV": "Ⅳ", "V": "Ⅴ",
                    "VI": "Ⅵ", "VII": "Ⅶ", "VIII": "Ⅷ", "IX": "Ⅸ", "X": "Ⅹ"
                }
                cur_norm = roman_map.get(cur.rstrip(".．"), cur.rstrip(".．"))

                def normalize_section_title(x):
                    body = x.replace(" ", "")
                    return spaced_title_map.get(body, x.replace(" ", ""))

                # 바로 다음 줄에 제목이 있으면 결합
                if nxt and edu_section_title_pattern.fullmatch(nxt):
                    merged_lines.append(f"{cur_norm}. {normalize_section_title(nxt)}")
                    i += 2
                    continue

                # 빈 줄 하나를 건너뛰고 그 다음 줄에 제목이 있으면 결합
                if not nxt and i + 2 < len(lines):
                    nxt2 = lines[i + 2].strip()
                    if nxt2 and edu_section_title_pattern.fullmatch(nxt2):
                        merged_lines.append(f"{cur_norm}. {normalize_section_title(nxt2)}")
                        i += 3
                        continue

            # 3) 숫자 절 제목: 1 / 사업개요 -> 1. 사업개요
            if re.fullmatch(r"\d{1,2}", cur):
                if nxt and len(nxt) <= 40 and ":" not in nxt:
                    merged_lines.append(f"{cur}. {nxt}")
                    i += 2
                    continue
                elif not nxt and i + 2 < len(lines):
                    nxt2 = lines[i + 2].strip()
                    if nxt2 and len(nxt2) <= 40 and ":" not in nxt2:
                        merged_lines.append(f"{cur}. {nxt2}")
                        i += 3
                        continue

        merged_lines.append(cur)
        i += 1

    # --------------------------------------------------
    # 4) 줄 단위 세부 정제
    # --------------------------------------------------
    new_lines = []

    for idx, line in enumerate(merged_lines):
        s = line.strip()

        if not s:
            new_lines.append("")
            continue

        # 보호 줄 유지
        if protected_line_pattern.search(s):
            new_lines.append(s)
            continue

        # ----------------------------------------------
        # 4-1) 확실한 OCR/이미지 잔여물 제거
        # ----------------------------------------------
        if re.fullmatch(r"(세로|가로)\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue

        if re.fullmatch(r"[·•○◦\-_=~]{1,6}", s):
            continue

        # 단독 숫자 줄 제거
        if re.fullmatch(r"\d{1,2}", s):
            continue

        # 고립 노이즈 제거
        if s in {"-", "l", "I"}:
            continue

        # 중간에 튀어나온 목차 제거
        # 상단부 첫 목차는 남기고, 문서 중간 이후 반복되는 목차류만 제거
        if idx > 12 and s in {"목차", "<목차>", "< 목 차 >"}:
            continue

        # ----------------------------------------------
        # 4-2) 표제형 띄어쓰기 제한적 정리
        # ----------------------------------------------
        if (
            len(s) <= 35
            and ":" not in s
            and not re.search(r"\d{3,}", s)
            and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,25}[가-힣A-Za-z]", s)
        ):
            s = s.replace(" ", "")

        # 예: "Ⅰ. 사 업 개 요" -> "Ⅰ. 사업개요"
        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*)([가-힣A-Za-z](?:\s[가-힣A-Za-z]){1,25})$",
            lambda m: m.group(1) + m.group(2).replace(" ", ""),
            s
        )

        # "Ⅰ 개요" -> "Ⅰ. 개요"
        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+)\s+([가-힣A-Za-z].+)$",
            r"\1. \2",
            s
        )

        # "1 사업개요" -> "1. 사업개요"
        s = re.sub(
            r"^(\d{1,2})\s+([가-힣A-Za-z].+)$",
            r"\1. \2",
            s
        )

        def fix_spaced_korean_in_parens(m):
            inner = m.group(1)
            if (
                len(inner) <= 35
                and not re.search(r"\d{3,}", inner)
                and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,25}[가-힣A-Za-z]", inner)
            ):
                return f"({inner.replace(' ', '')})"
            return m.group(0)

        s = re.sub(
            r"\(((?:[가-힣A-Za-z]\s){1,25}[가-힣A-Za-z])\)",
            fix_spaced_korean_in_parens,
            s
        )

        # ----------------------------------------------
        # 4-3) 목차형 줄 끝 페이지 번호 제거
        # ----------------------------------------------
        is_toc_like = bool(re.match(
            r"""^(
                \[.*\]|
                <.*>|
                [ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*|
                제?\d+장\s+|
                \d+(\.\d+)*[\.\)]\s+|
                [가나다라마바사아자차카타파하][\.\)]\s+|
                (별지|별표|붙임|서식|참조)\s*\d*|
                (사업명|사업개요|사업범위|추진배경|기대효과|제안서|제안요청|제안안내|기타사항)
            )""",
            s,
            re.VERBOSE
        ))

        if is_toc_like:
            s = re.sub(r"\s*-\s*-?\s*\d{1,3}(?:\s*[xX])?\s*$", "", s)
            s = re.sub(r"\s*[·\.…]{3,}\s*\d{1,3}\s*$", "", s)

        new_lines.append(s)

    text = "\n".join(new_lines)

    # --------------------------------------------------
    # 5) 후처리
    # --------------------------------------------------
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

#### 4. 지자체(local)

In [287]:
def clean_text_local(text):
    """
    지자체(local) 전용 보수적 정제

    원칙
    - 본문 핵심 정보 보존 우선
    - 연락처/이메일/URL은 보호
    - 제목형 띄어쓰기만 제한적으로 정리
    - 2줄짜리 짧은 표제/장제목 일부 결합
    - 목차형 줄 끝 페이지번호 제거
    - 확실한 OCR/파싱 잔여물만 제거
    - 본문/표 의미 훼손 가능성이 있는 과도한 삭제는 지양
    """
    if pd.isna(text):
        return text

    raw = str(text).strip()
    if not raw:
        return raw

    # --------------------------------------------------
    # 0) 입력 전체가 보호 대상이면 그대로 반환
    # --------------------------------------------------
    full_protected_patterns = [
        r"^\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^(?:TEL|FAX)\s*:\s*\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
        r"^https?://\S+$",
    ]
    for pat in full_protected_patterns:
        if re.fullmatch(pat, raw, flags=re.IGNORECASE):
            return raw

    # --------------------------------------------------
    # 1) 공통 정제
    # --------------------------------------------------
    text = clean_text_common(raw)
    text = text.replace("㈜", "(주)")

    # --------------------------------------------------
    # 2) 줄 보호 패턴
    # --------------------------------------------------
    protected_line_pattern = re.compile(
        r"("
        r"^\d{2,4}-\d{2,4}-\d{3,4}$"
        r"|(?:TEL|FAX)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}"
        r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
        r"|https?://\S+"
        r")",
        re.IGNORECASE
    )

    # --------------------------------------------------
    # 3) 줄 결합
    # --------------------------------------------------
    lines = [line.strip() for line in text.split("\n")]
    merged_lines = []
    i = 0

    merge_pairs = {
        ("목", "차"): "목차",
        ("순", "서 -"): "순서",
        ("순", "서"): "순서",
        ("개", "요"): "개요",
        ("현", "황"): "현황",
        ("붙", "임"): "붙임",
        ("별", "표"): "별표",
        ("별", "지"): "별지",
        ("서", "식"): "서식",
        ("안", "내"): "안내",
        ("사", "업"): "사업",
    }

    while i < len(lines):
        cur = lines[i]

        if i + 1 < len(lines):
            nxt = lines[i + 1]

            if protected_line_pattern.search(cur) or protected_line_pattern.search(nxt):
                merged_lines.append(cur)
                i += 1
                continue

            merged = merge_pairs.get((cur, nxt))
            if merged:
                merged_lines.append(merged)
                i += 2
                continue

            # Ⅰ / 사업개요  -> Ⅰ 사업개요
            if re.fullmatch(r"[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+", cur) and re.fullmatch(
                r"(?:[가-힣A-Za-z]\s){0,25}[가-힣A-Za-z]", nxt
            ):
                merged_lines.append(f"{cur} {nxt.replace(' ', '')}")
                i += 2
                continue

            # 1 / 사업일반 -> 1 사업일반
            if re.fullmatch(r"\d{1,2}(?:\.\d{1,2})?", cur) and len(nxt) <= 35 and ":" not in nxt:
                merged_lines.append(f"{cur} {nxt}")
                i += 2
                continue

        merged_lines.append(cur)
        i += 1

    # --------------------------------------------------
    # 4) 줄 단위 정제
    # --------------------------------------------------
    new_lines = []

    for line in merged_lines:
        s = line.strip()

        if not s:
            new_lines.append("")
            continue

        # 보호 줄 유지
        if protected_line_pattern.search(s):
            new_lines.append(s)
            continue

        # ----------------------------------------------
        # 4-1) 확실한 OCR/파싱 잔여물 제거
        # ----------------------------------------------
        if re.fullmatch(r"세로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue
        if re.fullmatch(r"가로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue

        # 단독 노이즈 기호
        if s in {"ㅣ", "|", "ᚺ"}:
            continue

        # 의미 없는 기호성 단독 줄만 제거
        if re.fullmatch(r"[·•○◦\-_=~]{1,6}", s):
            continue

        # 숫자만 있는 매우 짧은 줄 제거 (고립 페이지번호 가능성)
        if re.fullmatch(r"\d{1,2}", s):
            continue

        # '순 서 -', '순서 -' 같은 줄 정리
        if re.fullmatch(r"순\s*서\s*-\s*", s):
            s = "순서"

        # 줄 끝에 붙은 이상 문자 제거
        s = re.sub(r"\s*[ㅣᚺ]+\s*$", "", s)

        # ----------------------------------------------
        # 4-2) 제목형 띄어쓰기 제한 정리
        # ----------------------------------------------
        # 예: "제 안 요 청 서" -> "제안요청서"
        if (
            len(s) <= 40
            and ":" not in s
            and not re.search(r"\d{3,}", s)
            and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,30}[가-힣A-Za-z]", s)
        ):
            s = s.replace(" ", "")

        # 예: "Ⅰ. 사 업 개 요" -> "Ⅰ. 사업개요"
        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*)([가-힣A-Za-z](?:\s[가-힣A-Za-z]){1,30})$",
            lambda m: m.group(1) + m.group(2).replace(" ", ""),
            s
        )

        # 괄호 안 제목형 띄어쓰기
        def fix_spaced_korean_in_parens(m):
            inner = m.group(1)
            if (
                len(inner) <= 40
                and not re.search(r"\d{3,}", inner)
                and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,30}[가-힣A-Za-z]", inner)
            ):
                return f"({inner.replace(' ', '')})"
            return m.group(0)

        s = re.sub(
            r"\(((?:[가-힣A-Za-z]\s){1,30}[가-힣A-Za-z])\)",
            fix_spaced_korean_in_parens,
            s
        )

        # ----------------------------------------------
        # 4-3) 목차형 줄 끝 페이지번호 제거
        # ----------------------------------------------
        is_toc_like = bool(re.match(
            r"""^(
                \[.*\]|
                <.*>|
                [ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*|
                제?\d+장\s+|
                \d+(\.\d+)*[\.\)]\s+|
                [가나다라마바사아자차카타파하][\.\)]\s+|
                (별지|별표|붙임|서식)\s*\d*
            )""",
            s,
            re.VERBOSE
        ))

        if is_toc_like:
            s = re.sub(r"\s*-\s*-?\s*\d{1,3}(?:\s*[xX])?\s*$", "", s)
            s = re.sub(r"\s*[·\.…]{3,}\s*\d{1,3}\s*$", "", s)

        new_lines.append(s)

    text = "\n".join(new_lines)

    # --------------------------------------------------
    # 5) 빈 줄 정리
    # --------------------------------------------------
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

#### 5. 재단/협회/비영리(nonprofit)

In [288]:
def clean_text_nonprofit(text):
    """
    재단/협회/비영리(nonprofit) 전용 보수적 정제
    """
    if pd.isna(text):
        return text

    raw = str(text).strip()
    if not raw:
        return raw

    # --------------------------------------------------
    # 0) 입력 전체가 보호 대상이면 그대로 반환
    # --------------------------------------------------
    full_protected_patterns = [
        r"^\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^(?:TEL|FAX)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
        r"^https?://\S+$",
    ]
    for pat in full_protected_patterns:
        if re.fullmatch(pat, raw, flags=re.IGNORECASE):
            return raw

    # --------------------------------------------------
    # 1) 공통 정제
    # --------------------------------------------------
    text = clean_text_common(raw)
    text = text.replace("㈜", "(주)")
    text = text.replace("（", "(").replace("）", ")")
    text = re.sub(r"[–—−]", "-", text)

    # --------------------------------------------------
    # 2) 줄 보호 패턴
    # --------------------------------------------------
    protected_line_pattern = re.compile(
        r"("
        r"^\d{2,4}-\d{2,4}-\d{3,4}$"
        r"|(?:TEL|FAX)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}"
        r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
        r"|https?://\S+"
        r")",
        re.IGNORECASE
    )

    # --------------------------------------------------
    # 3) 줄 결합
    # --------------------------------------------------
    lines = [line.strip() for line in text.split("\n")]
    merged_lines = []
    i = 0

    merge_pairs = {
        ("목", "차"): "목차",
        ("순", "서"): "순서",
        ("순", "서 -"): "순서",
        ("개", "요"): "개요",
        ("현", "황"): "현황",
        ("붙", "임"): "붙임",
        ("별", "표"): "별표",
        ("별", "지"): "별지",
        ("서", "식"): "서식",
        ("안", "내"): "안내",
        ("사", "업"): "사업",
        ("과", "업"): "과업",
    }

    roman_only_pattern = re.compile(
        r"^(?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+|[IVXLC]+)[\.．]?$"
    )

    while i < len(lines):
        cur = lines[i]

        if not cur:
            merged_lines.append("")
            i += 1
            continue

        if i + 1 < len(lines):
            nxt = lines[i + 1]

            if protected_line_pattern.search(cur) or protected_line_pattern.search(nxt):
                merged_lines.append(cur)
                i += 1
                continue

            merged = merge_pairs.get((cur, nxt))
            if merged:
                merged_lines.append(merged)
                i += 2
                continue

            # 로마숫자 장 제목 결합
            if roman_only_pattern.fullmatch(cur):
                roman_map = {
                    "I": "Ⅰ", "II": "Ⅱ", "III": "Ⅲ", "IV": "Ⅳ", "V": "Ⅴ",
                    "VI": "Ⅵ", "VII": "Ⅶ", "VIII": "Ⅷ", "IX": "Ⅸ", "X": "Ⅹ"
                }
                cur_norm = roman_map.get(cur.rstrip(".．"), cur.rstrip(".．"))

                if nxt and len(nxt) <= 35 and ":" not in nxt:
                    merged_lines.append(f"{cur_norm}. {nxt}")
                    i += 2
                    continue
                elif i + 2 < len(lines):
                    nxt2 = lines[i + 2].strip()
                    if nxt2 and len(nxt2) <= 35 and ":" not in nxt2:
                        merged_lines.append(f"{cur_norm}. {nxt2}")
                        i += 3
                        continue

            # 숫자 절 제목 결합
            # 본문의 "1 / 사업목적"만 허용하고,
            # BIFF 같은 목차 페이지번호 줄은 막는다.
            if re.fullmatch(r"\d{1,2}(?:\.\d{1,2})?", cur):
                prev_line = merged_lines[-1].strip() if merged_lines else ""
                prev_is_roman_or_chapter = bool(
                    re.match(r"^(?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?|제?\d+장)\b", prev_line)
                )
                prev_is_blank = (prev_line == "")

                next_is_roman_or_chapter = bool(
                    re.match(r"^(?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?|제?\d+장)\b", nxt)
                )
                next_is_numbered = bool(
                    re.match(r"^\d{1,2}(?:\.\d{1,2})?[\.\)]\s*", nxt)
                )

                # 본문 번호 결합은:
                # - 앞줄이 비어 있거나 로마숫자/장 제목일 때만
                # - 다음 줄이 짧은 제목일 때만
                # - 다음 줄이 또 다른 번호/장 제목이면 금지
                if (
                    (prev_is_blank or prev_is_roman_or_chapter)
                    and nxt
                    and len(nxt) <= 40
                    and ":" not in nxt
                    and not next_is_roman_or_chapter
                    and not next_is_numbered
                ):
                    merged_lines.append(f"{cur}. {nxt}")
                    i += 2
                    continue

                elif i + 2 < len(lines) and not nxt:
                    nxt2 = lines[i + 2].strip()
                    nxt2_is_roman_or_chapter = bool(
                        re.match(r"^(?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?|제?\d+장)\b", nxt2)
                    )
                    nxt2_is_numbered = bool(
                        re.match(r"^\d{1,2}(?:\.\d{1,2})?[\.\)]\s*", nxt2)
                    )

                    if (
                        (prev_is_blank or prev_is_roman_or_chapter)
                        and nxt2
                        and len(nxt2) <= 40
                        and ":" not in nxt2
                        and not nxt2_is_roman_or_chapter
                        and not nxt2_is_numbered
                    ):
                        merged_lines.append(f"{cur}. {nxt2}")
                        i += 3
                        continue

        merged_lines.append(cur)
        i += 1

    # --------------------------------------------------
    # 4) 줄 단위 정제
    # --------------------------------------------------
    new_lines = []

    for line in merged_lines:
        s = line.strip()

        if not s:
            new_lines.append("")
            continue

        if protected_line_pattern.search(s):
            new_lines.append(s)
            continue

        if re.fullmatch(r"세로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue
        if re.fullmatch(r"가로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue

        if s in {"ㅣ", "|", "ᚺ"}:
            continue

        if re.fullmatch(r"신\s*-\s*", s):
            continue

        if re.fullmatch(r"[·•○◦\-_=~]{1,6}", s):
            continue

        if re.fullmatch(r"\d{1,2}", s):
            continue

        if re.fullmatch(r"순\s*서\s*-\s*", s):
            s = "순서"

        s = re.sub(r"\s*[ㅣᚺ]+\s*$", "", s)

        if (
            len(s) <= 45
            and ":" not in s
            and not re.search(r"\d{3,}", s)
            and re.fullmatch(r"(?:[가-힣A-Za-z&]\s){1,35}[가-힣A-Za-z&]", s)
        ):
            s = s.replace(" ", "")

        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*)([가-힣A-Za-z&](?:\s[가-힣A-Za-z&]){1,35})$",
            lambda m: m.group(1) + m.group(2).replace(" ", ""),
            s
        )

        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+)\s+([가-힣A-Za-z].+)$",
            r"\1. \2",
            s
        )

        s = re.sub(
            r"^(\d{1,2}(?:\.\d{1,2})?)\s+([가-힣A-Za-z].+)$",
            r"\1. \2",
            s
        )

        def fix_spaced_korean_in_parens(m):
            inner = m.group(1)
            if (
                len(inner) <= 45
                and not re.search(r"\d{3,}", inner)
                and re.fullmatch(r"(?:[가-힣A-Za-z&]\s){1,35}[가-힣A-Za-z&]", inner)
            ):
                return f"({inner.replace(' ', '')})"
            return m.group(0)

        s = re.sub(
            r"\(((?:[가-힣A-Za-z&]\s){1,35}[가-힣A-Za-z&])\)",
            fix_spaced_korean_in_parens,
            s
        )

        is_toc_like = bool(re.match(
            r"""^(
                \*?\s*\[.*\]|
                <.*>|
                [ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*|
                제?\d+장\s+|
                \d+(\.\d+)*[\.\)]\s+|
                [가나다라마바사아자차카타파하][\.\)]\s+|
                (별지|별표|붙임|서식)\s*\d*
            )""",
            s,
            re.VERBOSE
        ))

        if is_toc_like:
            s = re.sub(r"\s*-\s*-?\s*\d{1,3}(?:\s*[xX])?\s*$", "", s)
            s = re.sub(r"\s*[·\.…]{3,}\s*\d{1,3}\s*$", "", s)

        new_lines.append(s)

    text = "\n".join(new_lines)

    # --------------------------------------------------
    # 5) 후처리
    # --------------------------------------------------
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

#### 6. 중앙행정기관(central)

In [289]:
def clean_text_central(text):
    """
    중앙행정기관(central) 전용 보수적 정제

    원칙
    - 본문 핵심 정보 보존 우선
    - 연락처/이메일/URL은 보호
    - 제목형 띄어쓰기만 제한적으로 정리
    - 2줄짜리 짧은 표제/장제목 일부 결합
    - 목차형 줄 끝 페이지번호 제거
    - 확실한 OCR/파싱 잔여물만 제거
    - 본문/표 의미 훼손 가능성이 있는 과도한 삭제는 지양
    """
    if pd.isna(text):
        return text

    raw = str(text).strip()
    if not raw:
        return raw

    # --------------------------------------------------
    # 0) 입력 전체가 보호 대상이면 그대로 반환
    # --------------------------------------------------
    full_protected_patterns = [
        r"^\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^(?:TEL|FAX|전화|팩스)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
        r"^https?://\S+$",
    ]
    for pat in full_protected_patterns:
        if re.fullmatch(pat, raw, flags=re.IGNORECASE):
            return raw

    # --------------------------------------------------
    # 1) 공통 정제
    # --------------------------------------------------
    text = clean_text_common(raw)
    text = text.replace("㈜", "(주)")
    text = text.replace("ῼ", " ")

    # --------------------------------------------------
    # 2) 줄 보호 패턴
    # --------------------------------------------------
    protected_line_pattern = re.compile(
        r"("
        r"^\d{2,4}-\d{2,4}-\d{3,4}$"
        r"|(?:TEL|FAX|전화|팩스)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}"
        r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
        r"|https?://\S+"
        r")",
        re.IGNORECASE
    )

    # --------------------------------------------------
    # 3) 줄 결합
    # --------------------------------------------------
    lines = [line.strip() for line in text.split("\n")]
    merged_lines = []
    i = 0

    merge_pairs = {
        ("목", "차"): "목차",
        ("개", "요"): "개요",
        ("현", "황"): "현황",
        ("붙", "임"): "붙임",
        ("별", "지"): "별지",
        ("별", "표"): "별표",
        ("서", "식"): "서식",
        ("안", "내"): "안내",
        ("사", "업"): "사업",
        ("요", "구"): "요구",
        ("내", "용"): "내용",
    }

    while i < len(lines):
        cur = lines[i]

        if i + 1 < len(lines):
            nxt = lines[i + 1]

            if protected_line_pattern.search(cur) or protected_line_pattern.search(nxt):
                merged_lines.append(cur)
                i += 1
                continue

            merged = merge_pairs.get((cur, nxt))
            if merged:
                merged_lines.append(merged)
                i += 2
                continue

            # Ⅰ / 사 업 개 요  -> Ⅰ 사업개요
            if re.fullmatch(r"[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+", cur) and re.fullmatch(
                r"(?:[가-힣A-Za-z]\s){0,30}[가-힣A-Za-z]", nxt
            ):
                merged_lines.append(f"{cur} {nxt.replace(' ', '')}")
                i += 2
                continue

            # 1 / 사 업 일 반 -> 1 사업일반
            if re.fullmatch(r"\d{1,2}(?:\.\d{1,2})?", cur) and re.fullmatch(
                r"(?:[가-힣A-Za-z]\s){0,30}[가-힣A-Za-z]", nxt
            ):
                merged_lines.append(f"{cur} {nxt.replace(' ', '')}")
                i += 2
                continue

        merged_lines.append(cur)
        i += 1

    # --------------------------------------------------
    # 4) 줄 단위 정제
    # --------------------------------------------------
    new_lines = []

    for line in merged_lines:
        s = line.strip()

        if not s:
            new_lines.append("")
            continue

        # 보호 줄 유지
        if protected_line_pattern.search(s):
            new_lines.append(s)
            continue

        # ----------------------------------------------
        # 4-1) 확실한 장식/노이즈만 제거
        # ----------------------------------------------
        # 예: • • • 목 차 • •  -> 목차
        if re.fullmatch(r"[•·\s]*목\s*차[•·\s]*", s):
            new_lines.append("목차")
            continue

        # 단독 장식 기호 줄
        if re.fullmatch(r"[•·○◦\-_=~]{1,10}", s):
            continue

        # pixel 잔여물
        if re.fullmatch(r"세로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue
        if re.fullmatch(r"가로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue

        # 숫자만 있는 매우 짧은 줄 제거 (고립 페이지번호 가능성)
        if re.fullmatch(r"\d{1,2}", s):
            continue

        # ----------------------------------------------
        # 4-2) 제목형 띄어쓰기 제한 정리
        # ----------------------------------------------
        # 예: 제 안 요 청 서 -> 제안요청서
        if (
            len(s) <= 45
            and ":" not in s
            and not re.search(r"\d{3,}", s)
            and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,35}[가-힣A-Za-z]", s)
        ):
            s = s.replace(" ", "")

        # 예: Ⅰ. 사 업 개 요 -> Ⅰ. 사업개요
        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*)([가-힣A-Za-z](?:\s[가-힣A-Za-z]){1,35})$",
            lambda m: m.group(1) + m.group(2).replace(" ", ""),
            s
        )

        # 괄호 안 제목형 띄어쓰기
        def fix_spaced_korean_in_parens(m):
            inner = m.group(1)
            if (
                len(inner) <= 45
                and not re.search(r"\d{3,}", inner)
                and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,35}[가-힣A-Za-z]", inner)
            ):
                return f"({inner.replace(' ', '')})"
            return m.group(0)

        s = re.sub(
            r"\(((?:[가-힣A-Za-z]\s){1,35}[가-힣A-Za-z])\)",
            fix_spaced_korean_in_parens,
            s
        )

        # ----------------------------------------------
        # 4-3) 목차형 줄 끝 페이지번호 제거
        # ----------------------------------------------
        is_toc_like = bool(re.match(
            r"""^(
                \[.*\]|
                <.*>|
                [ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*|
                제?\d+장\s+|
                \d+(\.\d+)*[\.\)]\s+|
                [가나다라마바사아자차카타파하][\.\)]\s+|
                (붙임|별지|별표|서식)\s*\d*
            )""",
            s,
            re.VERBOSE
        ))

        if is_toc_like:
            s = re.sub(r"\s*-\s*-?\s*\d{1,3}(?:\s*[xX])?\s*$", "", s)
            s = re.sub(r"\s*[·\.…]{3,}\s*\d{1,3}\s*$", "", s)

        new_lines.append(s)

    text = "\n".join(new_lines)

    # --------------------------------------------------
    # 5) 빈 줄 정리
    # --------------------------------------------------
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

#### 7. 민간기업(company)

In [290]:
def clean_text_company(text):
    """
    민간기업(company) 전용 보수적 정제

    원칙
    - 본문 핵심 정보 보존 우선
    - 연락처/이메일/URL은 보호
    - 제목형 띄어쓰기만 제한적으로 정리
    - 2줄짜리 짧은 표제/장제목 일부 결합
    - 목차형 줄 끝 페이지번호 제거
    - 목차 안의 확실한 잡음 문자만 제거
    - 요구사항 코드/기술용어/약어는 보존
    """
    if pd.isna(text):
        return text

    raw = str(text).strip()
    if not raw:
        return raw

    # --------------------------------------------------
    # 0) 입력 전체가 보호 대상이면 그대로 반환
    # --------------------------------------------------
    full_protected_patterns = [
        r"^\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^(?:TEL|FAX|전화|팩스)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
        r"^https?://\S+$",
    ]
    for pat in full_protected_patterns:
        if re.fullmatch(pat, raw, flags=re.IGNORECASE):
            return raw

    # --------------------------------------------------
    # 1) 공통 정제
    # --------------------------------------------------
    text = clean_text_common(raw)
    text = text.replace("㈜", "(주)")
    text = text.replace("（", "(").replace("）", ")")
    text = re.sub(r"[–—−]", "-", text)

    # --------------------------------------------------
    # 2) 줄 보호 패턴
    # --------------------------------------------------
    protected_line_pattern = re.compile(
        r"("
        r"^\d{2,4}-\d{2,4}-\d{3,4}$"
        r"|(?:TEL|FAX|전화|팩스)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}"
        r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
        r"|https?://\S+"
        r")",
        re.IGNORECASE
    )

    # --------------------------------------------------
    # 3) 줄 결합
    # --------------------------------------------------
    lines = [line.strip() for line in text.split("\n")]
    merged_lines = []
    i = 0

    merge_pairs = {
        ("목", "차"): "목차",
        ("개", "요"): "개요",
        ("현", "황"): "현황",
        ("붙", "임"): "붙임",
        ("별", "표"): "별표",
        ("별", "지"): "별지",
        ("서", "식"): "서식",
        ("안", "내"): "안내",
        ("사", "업"): "사업",
        ("과", "업"): "과업",
        ("요", "구"): "요구",
        ("내", "용"): "내용",
    }

    roman_only_pattern = re.compile(
        r"^(?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+|[IVXLC]+)[\.．]?$"
    )

    while i < len(lines):
        cur = lines[i]

        if not cur:
            merged_lines.append("")
            i += 1
            continue

        if i + 1 < len(lines):
            nxt = lines[i + 1]

            if protected_line_pattern.search(cur) or protected_line_pattern.search(nxt):
                merged_lines.append(cur)
                i += 1
                continue

            merged = merge_pairs.get((cur, nxt))
            if merged:
                merged_lines.append(merged)
                i += 2
                continue

            # Ⅰ / 사업개요 -> Ⅰ. 사업개요
            # Ⅰ / 빈줄 / 사업개요 -> Ⅰ. 사업개요
            if roman_only_pattern.fullmatch(cur):
                roman_map = {
                    "I": "Ⅰ", "II": "Ⅱ", "III": "Ⅲ", "IV": "Ⅳ", "V": "Ⅴ",
                    "VI": "Ⅵ", "VII": "Ⅶ", "VIII": "Ⅷ", "IX": "Ⅸ", "X": "Ⅹ"
                }
                cur_norm = roman_map.get(cur.rstrip(".．"), cur.rstrip(".．"))

                if nxt and len(nxt) <= 35 and ":" not in nxt:
                    merged_lines.append(f"{cur_norm}. {nxt}")
                    i += 2
                    continue
                elif i + 2 < len(lines):
                    nxt2 = lines[i + 2].strip()
                    if nxt2 and len(nxt2) <= 35 and ":" not in nxt2:
                        merged_lines.append(f"{cur_norm}. {nxt2}")
                        i += 3
                        continue

            # 1 / 추진배경 -> 1. 추진배경
            if re.fullmatch(r"\d{1,2}(?:\.\d{1,2})?", cur):
                if nxt and len(nxt) <= 45 and ":" not in nxt:
                    merged_lines.append(f"{cur}. {nxt}")
                    i += 2
                    continue
                elif i + 2 < len(lines) and not nxt:
                    nxt2 = lines[i + 2].strip()
                    if nxt2 and len(nxt2) <= 45 and ":" not in nxt2:
                        merged_lines.append(f"{cur}. {nxt2}")
                        i += 3
                        continue

        merged_lines.append(cur)
        i += 1

    # --------------------------------------------------
    # 4) 줄 단위 정제
    # --------------------------------------------------
    new_lines = []

    for line in merged_lines:
        s = line.strip()

        if not s:
            new_lines.append("")
            continue

        if protected_line_pattern.search(s):
            new_lines.append(s)
            continue

        # ----------------------------------------------
        # 4-1) 확실한 잔여물만 제거
        # ----------------------------------------------
        if re.fullmatch(r"세로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue
        if re.fullmatch(r"가로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue

        # 단독 노이즈
        if s in {"ㅣ", "|", "ᚺ", "ḿ", "⁻"}:
            continue

        # 의미 없는 장식 줄
        if re.fullmatch(r"[·•○◦\-_=~]{1,10}", s):
            continue

        # 숫자만 있는 매우 짧은 줄 제거 (페이지번호 가능성)
        if re.fullmatch(r"\d{1,2}", s):
            continue

        # 줄 끝 이상 문자 제거
        s = re.sub(r"\s*[ḿ⁻ᚺ]+\s*$", "", s)

        # ----------------------------------------------
        # 4-2) 제목형 띄어쓰기 제한 정리
        # ----------------------------------------------
        if (
            len(s) <= 50
            and ":" not in s
            and not re.search(r"\d{3,}", s)
            and re.fullmatch(r"(?:[가-힣A-Za-z&/]\s){1,40}[가-힣A-Za-z&/]", s)
        ):
            s = s.replace(" ", "")

        # 예: Ⅰ. 사 업 개 요 -> Ⅰ. 사업개요
        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*)([가-힣A-Za-z&/](?:\s[가-힣A-Za-z&/]){1,40})$",
            lambda m: m.group(1) + m.group(2).replace(" ", ""),
            s
        )

        # 예: Ⅰ 사업개요 -> Ⅰ. 사업개요
        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+)\s+([가-힣A-Za-z].+)$",
            r"\1. \2",
            s
        )

        # 예: 1 추진배경 -> 1. 추진배경
        s = re.sub(
            r"^(\d{1,2}(?:\.\d{1,2})?)\s+([가-힣A-Za-z].+)$",
            r"\1. \2",
            s
        )

        # 괄호 안 제목형 띄어쓰기
        def fix_spaced_korean_in_parens(m):
            inner = m.group(1)
            if (
                len(inner) <= 50
                and not re.search(r"\d{3,}", inner)
                and re.fullmatch(r"(?:[가-힣A-Za-z&/]\s){1,40}[가-힣A-Za-z&/]", inner)
            ):
                return f"({inner.replace(' ', '')})"
            return m.group(0)

        s = re.sub(
            r"\(((?:[가-힣A-Za-z&/]\s){1,40}[가-힣A-Za-z&/])\)",
            fix_spaced_korean_in_parens,
            s
        )

        # ----------------------------------------------
        # 4-3) 목차형 줄의 잡음 문자 정리
        # ----------------------------------------------
        is_toc_like = bool(re.match(
            r"""^(
                \[.*\]|
                <.*>|
                [ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*|
                제?\d+장\s+|
                \d+(\.\d+)*[\.\)]\s+|
                [가나다라마바사아자차카타파하][\.\)]\s+|
                (붙임|별지|별표|서식)\s*\d*|
                (공통|기능|인터페이스|테스트|보안|품질|제약사항|프로젝트)
            )""",
            s,
            re.VERBOSE
        ))

        if is_toc_like:
            # 목차 안에서만 잡음 제거
            s = re.sub(r"\s*[ḿ⁻]+\s*", " ", s)
            s = re.sub(r"\s*-\s*-\s*(\d{1,3})\s*$", r" - \1", s)
            s = re.sub(r"\s*-\s*-?\s*\d{1,3}(?:\s*[xX])?\s*$", "", s)
            s = re.sub(r"\s*[·\.…]{3,}\s*\d{1,3}\s*$", "", s)

        new_lines.append(s)

    text = "\n".join(new_lines)

    # --------------------------------------------------
    # 5) 빈 줄 정리
    # --------------------------------------------------
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

#### 섹터별 정제함수 적용

In [291]:
# 1. 공기업/공공기관(public)
mask_public_hwp = (df["기관 섹터"] == "공기업/공공기관(public)") & is_hwp
df.loc[mask_public_hwp, "정제텍스트"] = df.loc[mask_public_hwp, "공통정제텍스트"].apply(clean_text_public)

# 2. 연구기관(research)
mask_research_hwp = (df["기관 섹터"] == "연구기관(research)") & is_hwp
df.loc[mask_research_hwp, "정제텍스트"] = df.loc[mask_research_hwp, "공통정제텍스트"].apply(clean_text_research)

# 3. 대학/교육기관(education)
mask_education_hwp = (df["기관 섹터"] == "대학/교육기관(education)") & is_hwp
df.loc[mask_education_hwp, "정제텍스트"] = df.loc[mask_education_hwp, "공통정제텍스트"].apply(clean_text_education)

# 4. 지자체(local)
mask_local_hwp = (df["기관 섹터"] == "지자체(local)") & is_hwp
df.loc[mask_local_hwp, "정제텍스트"] = df.loc[mask_local_hwp, "공통정제텍스트"].apply(clean_text_local)

# 5. 재단/협회/비영리(nonprofit)
mask_nonprofit_hwp = (df["기관 섹터"] == "재단/협회/비영리(nonprofit)") & is_hwp
df.loc[mask_nonprofit_hwp, "정제텍스트"] = df.loc[mask_nonprofit_hwp, "공통정제텍스트"].apply(clean_text_nonprofit)

# 6. 중앙행정기관(central)
mask_central_hwp = (df["기관 섹터"] == "중앙행정기관(central)") & is_hwp
df.loc[mask_central_hwp, "정제텍스트"] = df.loc[mask_central_hwp, "공통정제텍스트"].apply(clean_text_central)

# 7. 민간기업(company)
mask_company_hwp = (df["기관 섹터"] == "민간기업(company)") & is_hwp
df.loc[mask_company_hwp, "정제텍스트"] = df.loc[mask_company_hwp, "공통정제텍스트"].apply(clean_text_company)

In [292]:
df.columns

Index(['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일',
       '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '텍스트', '텍스트길이', '기관 섹터', '공통정제텍스트',
       '정제텍스트'],
      dtype='str')

In [293]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 16 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   공고 번호      82 non-null     str    
 1   공고 차수      82 non-null     float64
 2   사업명        100 non-null    str    
 3   사업 금액      99 non-null     float64
 4   발주 기관      100 non-null    str    
 5   공개 일자      100 non-null    str    
 6   입찰 참여 시작일  74 non-null     str    
 7   입찰 참여 마감일  92 non-null     str    
 8   사업 요약      100 non-null    str    
 9   파일형식       100 non-null    str    
 10  파일명        100 non-null    str    
 11  텍스트        100 non-null    str    
 12  텍스트길이      7 non-null      float64
 13  기관 섹터      100 non-null    str    
 14  공통정제텍스트    100 non-null    str    
 15  정제텍스트      100 non-null    str    
dtypes: float64(3), str(13)
memory usage: 4.3 MB


In [294]:
output_path = "/home/bidcoin/df_hwp_v2.csv"
df.to_csv(output_path, index=False, encoding="utf-8")

print(f"저장 완료: {output_path}")

저장 완료: /home/bidcoin/df_hwp_v2.csv


## (2)데이터 합치기(pdf, hwp)

### (2-1)정제후 pdf 불러오기

In [295]:
# pdf
df_pdf = pd.read_csv("/home/bidcoin/df_pdf_v2.csv", encoding="utf-8")
df_pdf.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   공고 번호      4 non-null      str    
 1   공고 차수      4 non-null      float64
 2   사업명        6 non-null      str    
 3   사업 금액      6 non-null      float64
 4   발주 기관      6 non-null      str    
 5   공개 일자      6 non-null      str    
 6   입찰 참여 시작일  4 non-null      str    
 7   입찰 참여 마감일  5 non-null      str    
 8   사업 요약      6 non-null      str    
 9   파일형식       6 non-null      str    
 10  파일명        6 non-null      str    
 11  텍스트        6 non-null      str    
 12  텍스트길이      1 non-null      float64
dtypes: float64(3), str(10)
memory usage: 256.4 KB


In [296]:
df_pdf.shape

(6, 13)

### (2-2)정제후 hwp 불러오기

In [297]:
# hwp
df_hwp = pd.read_csv("/home/bidcoin/df_hwp_v2.csv", encoding="utf-8")
df_hwp.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 16 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   공고 번호      82 non-null     str    
 1   공고 차수      82 non-null     float64
 2   사업명        100 non-null    str    
 3   사업 금액      99 non-null     float64
 4   발주 기관      100 non-null    str    
 5   공개 일자      100 non-null    str    
 6   입찰 참여 시작일  74 non-null     str    
 7   입찰 참여 마감일  92 non-null     str    
 8   사업 요약      100 non-null    str    
 9   파일형식       100 non-null    str    
 10  파일명        100 non-null    str    
 11  텍스트        100 non-null    str    
 12  텍스트길이      7 non-null      float64
 13  기관 섹터      100 non-null    str    
 14  공통정제텍스트    100 non-null    str    
 15  정제텍스트      100 non-null    str    
dtypes: float64(3), str(13)
memory usage: 4.3 MB


In [298]:
df_hwp.columns

Index(['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일',
       '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '텍스트', '텍스트길이', '기관 섹터', '공통정제텍스트',
       '정제텍스트'],
      dtype='str')

In [299]:
df_hwp.shape

(100, 16)

In [300]:
# hwp만 골라내기
df_hwp = df_hwp[df_hwp["파일형식"] == "hwp"].copy()
df_hwp.shape

(94, 16)

#### hwp 정제텍스트->텍스트 덮어씌우기 (pdf랑 형식 맞추기)

In [301]:
df_hwp["텍스트"] = df_hwp["정제텍스트"]

In [302]:
df_hwp = df_hwp[
    [
        "공고 번호",
        "공고 차수",
        "사업명",
        "사업 금액",
        "발주 기관",
        "공개 일자",
        "입찰 참여 시작일",
        "입찰 참여 마감일",
        "사업 요약",
        "파일형식",
        "파일명",
        "텍스트",
        "텍스트길이",
    ]
].copy()

In [303]:
df_hwp.shape

(94, 13)

### (2-3)concat

In [304]:
df_merge = pd.concat([df_pdf, df_hwp], ignore_index=True)

In [305]:
df_merge.shape

(100, 13)

In [306]:
output_path = "/home/bidcoin/df_merge_v2.csv"
df_merge.to_csv(output_path, index=False, encoding="utf-8")

print(f"저장 완료: {output_path}")

저장 완료: /home/bidcoin/df_merge_v2.csv


## (3)결측처리

In [307]:
df2 = pd.read_csv("/home/bidcoin/df_merge_v2.csv", encoding="utf-8")
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   공고 번호      82 non-null     str    
 1   공고 차수      82 non-null     float64
 2   사업명        100 non-null    str    
 3   사업 금액      99 non-null     float64
 4   발주 기관      100 non-null    str    
 5   공개 일자      100 non-null    str    
 6   입찰 참여 시작일  74 non-null     str    
 7   입찰 참여 마감일  92 non-null     str    
 8   사업 요약      100 non-null    str    
 9   파일형식       100 non-null    str    
 10  파일명        100 non-null    str    
 11  텍스트        100 non-null    str    
 12  텍스트길이      7 non-null      float64
dtypes: float64(3), str(10)
memory usage: 1.5 MB


In [308]:
df2.columns

Index(['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일',
       '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '텍스트', '텍스트길이'],
      dtype='str')

### (3-1)입찰 참여 시작일

In [309]:
cols = ["파일명", "공개 일자", "입찰 참여 시작일", "입찰 참여 마감일"]

missing_start = df2.loc[
    df2["입찰 참여 시작일"].isna(),
    cols
].copy()

missing_start

,파일명,공개 일자,입찰 참여 시작일,입찰 참여 마감일
1,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,2023-06-20 00:00:00,NaN,NaN
4,대전대학교_대전대학교+2024학년도+다층적+융합+학습경험+플랫폼(MILE)+전.hw...,2024-11-27 11:36:47,NaN,2024-12-09 11:00:00
6,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,2024-10-04 13:51:23,NaN,2024-10-15 17:00:00
17,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,2024-05-02 00:00:00,NaN,NaN
18,한국수자원공사_건설통합시스템(CMS) 고도화.hwp,2024-05-31 00:00:00,NaN,NaN
21,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp,2024-09-10 16:31:49,NaN,2024-09-23 18:00:00
27,수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp,2025-02-11 10:27:38,NaN,2025-03-10 11:00:00
28,한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp,2024-05-13 00:00:00,NaN,NaN
29,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,2024-10-24 00:00:00,NaN,NaN
34,인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp,2025-01-24 19:56:15,NaN,2025-02-20 18:00:00


In [310]:
missing_start_list = missing_start["파일명"].tolist()
len(missing_start_list)

26

In [311]:
# 시작일은 마감일에 비해 중요도가 낮으니 그냥 공개일자로 대체
mask = df2["입찰 참여 시작일"].isna() & df2["공개 일자"].notna()

df2.loc[mask, "입찰 참여 시작일"] = df2.loc[mask, "공개 일자"]

In [312]:
# 확인
df2.loc[
    df2["파일명"].isin(missing_start_list),
    ["파일명", "공개 일자", "입찰 참여 시작일"]
]

,파일명,공개 일자,입찰 참여 시작일
1,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,2023-06-20 00:00:00,2023-06-20 00:00:00
4,대전대학교_대전대학교+2024학년도+다층적+융합+학습경험+플랫폼(MILE)+전.hw...,2024-11-27 11:36:47,2024-11-27 11:36:47
6,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,2024-10-04 13:51:23,2024-10-04 13:51:23
17,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,2024-05-02 00:00:00,2024-05-02 00:00:00
18,한국수자원공사_건설통합시스템(CMS) 고도화.hwp,2024-05-31 00:00:00,2024-05-31 00:00:00
21,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp,2024-09-10 16:31:49,2024-09-10 16:31:49
27,수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp,2025-02-11 10:27:38,2025-02-11 10:27:38
28,한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp,2024-05-13 00:00:00,2024-05-13 00:00:00
29,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,2024-10-24 00:00:00,2024-10-24 00:00:00
34,인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp,2025-01-24 19:56:15,2025-01-24 19:56:15


### (3-2)입찰 참여 마감일

In [313]:
cols2 = ["파일명", "공개 일자", "입찰 참여 시작일", "입찰 참여 마감일"]

missing_end = df2.loc[
    df2["입찰 참여 마감일"].isna(),
    cols2
].copy()

missing_end

,파일명,공개 일자,입찰 참여 시작일,입찰 참여 마감일
1,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,2023-06-20 00:00:00,2023-06-20 00:00:00,NaN
17,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,2024-05-02 00:00:00,2024-05-02 00:00:00,NaN
18,한국수자원공사_건설통합시스템(CMS) 고도화.hwp,2024-05-31 00:00:00,2024-05-31 00:00:00,NaN
28,한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp,2024-05-13 00:00:00,2024-05-13 00:00:00,NaN
29,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,2024-10-24 00:00:00,2024-10-24 00:00:00,NaN
47,BioIN_의료기기산업 종합정보시스템(정보관리기관) 기능개선 사업(2차).hwp,2024-09-05 00:00:00,2024-09-05 00:00:00,NaN
70,국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp,2024-08-13 00:00:00,2024-08-13 00:00:00,NaN
88,세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp,2021-10-08 00:00:00,2021-10-08 00:00:00,NaN


In [314]:
print(missing_end['파일명'])

1            서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf
17              경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
18                         한국수자원공사_건설통합시스템(CMS) 고도화.hwp
28             한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp
29    KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp
47         BioIN_의료기기산업 종합정보시스템(정보관리기관) 기능개선 사업(2차).hwp
70          국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp
88             세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp
Name: 파일명, dtype: str


In [315]:
# ( 입찰공고문 보라고 함...이건 제안요청서..)
# 12 : ...
# 13 : 2024년 05월 14일 11:00AM 마감
# 14 : ...
# 24 : ...
# 26 : ...
# 46 : ...
# 70 : ...
# 88 : ...

In [316]:
# 직접 입력
df2.loc[
    df2["파일명"] == "경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp",
    "입찰 참여 마감일"
] = "2024-05-14 11:00:00"

In [317]:
# 확인
mask = df2["파일명"] == "경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp"
df2.loc[mask, ["파일명", "입찰 참여 마감일"]]

,파일명,입찰 참여 마감일
17,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,2024-05-14 11:00:00


### (3-3)사업 금액

In [318]:
display(df2.loc[df2["사업 금액"].isna(), ["파일명","사업 금액"]])

,파일명,사업 금액
59,대한상공회의소_기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고.hwp,NaN


In [319]:
# 직접 입력
df2.loc[df2["사업 금액"].isna(), "사업 금액"] = 57000000
display(df2.loc[df2["파일명"] == "대한상공회의소_기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고.hwp", ["파일명", "사업 금액"]])

,파일명,사업 금액
59,대한상공회의소_기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고.hwp,57000000.0


In [320]:
df2['사업 금액'].min(), df2['사업 금액'].max()  # 0인거 채워넣어야 할듯

(np.float64(0.0), np.float64(14107009000.0))

In [321]:
df2[['파일명', '사업 금액']].nsmallest(10, '사업 금액')

,파일명,사업 금액
1,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,0.0
17,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,0.0
20,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,0.0
37,한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp,0.0
43,을지대학교_을지대학교 비교과시스템 개발.hwp,0.0
71,한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp,0.0
83,사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp,1.0
52,대검찰청_아태 사이버범죄 역량강화 허브(APC-HUB) 홈페이지 및 온라인 교.hwp,35750000.0
8,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,40000000.0
74,재단법인 광주연구원_광주정책연구아카이브(GPA) 시스템 개발.hwp,43000000.0


In [322]:
# 단위는 <원>
# 12 : 242,900,000원
# 13 : 1차년도(계약체결일 ~ 2025.04.30.) 200,000,000원, 2차년도(2025.05.01. ~ 2026.04.30.) 200,000,000원
# 16 : 470백만원
# 34 : 개발비 359백만원 + H/W 484백만원(VAT 포함) = 843백만원
# 41 : 비공개
# 71 : 비공개
# 83 : 비공개
# 52 : 35,750천원 = 35,750,000

In [323]:
# 직접 입력
df2.loc[df2["파일명"] == "서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf", "사업 금액"] = 242900000
df2.loc[df2["파일명"] == "경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp", "사업 금액"] = 200000000
df2.loc[df2["파일명"] == "한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp", "사업 금액"] = 470000000
df2.loc[df2["파일명"] == "한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp", "사업 금액"] = 843000000
df2.loc[df2["파일명"] == "사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp", "사업 금액"] = 0

In [324]:
# 확인
display(df2.loc[df2["파일명"] == "서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf", ["파일명", "사업 금액"]])
display(df2.loc[df2["파일명"] == "경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp", ["파일명", "사업 금액"]])
display(df2.loc[df2["파일명"] == "한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp", ["파일명", "사업 금액"]])
display(df2.loc[df2["파일명"] == "한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp", ["파일명", "사업 금액"]])
display(df2.loc[df2["파일명"] == "사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp", ["파일명", "사업 금액"]])

,파일명,사업 금액
1,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,242900000.0


,파일명,사업 금액
17,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,200000000.0


,파일명,사업 금액
20,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,470000000.0


,파일명,사업 금액
37,한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp,843000000.0


,파일명,사업 금액
83,사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp,0.0


### (3-4)공고 번호

In [325]:
df2[['공고 번호']].value_counts()

공고 번호        
R25BK00601569    1
20240404154      1
20241139040      1
20241120435      1
20241001798      1
20241002912      1
20240827859      1
20240430918      1
20240430896      1
R25BK00559883    1
R25BK00564730    1
20240821865      1
20240821893      1
20240812818      1
20240815487      1
20240910050      1
20240903676      1
20240903688      1
20240904268      1
R25BK00632049    1
R25BK00632248    1
R25BK00603644    1
R25BK00603533    1
R25BK00604826    1
R25BK00605617    1
20240523741      1
20240524568      1
20240413838      1
20240414353      1
20240345257      1
20241213403      1
20241138828      1
20241138864      1
20240723270      1
20240723668      1
20241130016      1
20241118572      1
20240531013      1
20240535775      1
20240539319      1
20240539643      1
20240611568      1
20240611774      1
20240605067      1
20240605351      1
20240605366      1
20241211469      1
20241217596      1
20241218257      1
20240541684      1
20240541779      1
20241207733      

In [326]:
df2.loc[df2['공고 번호'].astype(str).str.startswith('R'), ['파일명','발주 기관']]  # 9

,파일명,발주 기관
2,한국농어촌공사_아세안+3+식량안보정보시스템(AFSIS)+3단계+협력(캄보디아.hwp...,한국농어촌공사
11,한국전기안전공사_전기안전 관제시스템 보안 모듈 개발 용역.hwp,한국전기안전공사
12,재단법인충북연구원_GIS통계 기반 재난안전데이터 분석ㆍ관리 시스템 구.hwp,재단법인충북연구원
26,한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp,한국사회보장정보원
27,수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp,수협중앙회
31,국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp,국방과학연구소
32,인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp,인천광역시
33,"대한장애인체육회_2025년 전국장애인체육대회 전산 및 시스템, 홈페이지 .hwp",대한장애인체육회
34,인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp,인천광역시 동구


In [327]:
df2.loc[df2['공고 번호'].astype(str).str.startswith('2'), ['파일명','발주 기관']]  #73

,파일명,발주 기관
3,서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf,서울특별시
4,대전대학교_대전대학교+2024학년도+다층적+융합+학습경험+플랫폼(MILE)+전.hw...,대전대학교
5,기초과학연구원_2025년도 중이온가속기용 극저온시스템 운전 용역.pdf,기초과학연구원
6,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,한영대학
7,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,한국연구재단
8,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,한국생산기술연구원
9,인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp,인천광역시
10,경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp,경상북도 봉화군
13,재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp,재단법인스포츠윤리센터
14,국방과학연구소_대용량 자료전송시스템 고도화.hwp,국방과학연구소


In [328]:
agency_list = df2.loc[
    df2['공고 번호'].astype(str).str.startswith('R'),
    '발주 기관'
].drop_duplicates().tolist()

In [329]:
agency_list2 = df2.loc[
    df2['공고 번호'].astype(str).str.startswith('2'),
    '발주 기관'
].dropna().drop_duplicates().tolist()

In [330]:
overlap = set(agency_list) & set(agency_list2)
print(overlap)
print(len(overlap))

{'한국농어촌공사', '국방과학연구소', '인천광역시', '수협중앙회'}
4


In [331]:
# 발주 기관에 따른 건 아닌듯. 가짜임이 드러나는 값으로 채우고자함

In [332]:
mask = df2['공고 번호'].isna()
df2.loc[mask, '공고 번호'] = 'TEMP_' + df2.loc[mask].index.astype(str)

In [333]:
df2.loc[df2['공고 번호'].astype(str).str.startswith('TEMP_'), ['파일명', '공고 번호']]

,파일명,공고 번호
0,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,TEMP_0
1,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,TEMP_1
17,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,TEMP_17
18,한국수자원공사_건설통합시스템(CMS) 고도화.hwp,TEMP_18
19,국가과학기술지식정보서비스_통합정보시스템 고도화 용역.hwp,TEMP_19
20,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,TEMP_20
22,한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp,TEMP_22
28,한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp,TEMP_28
29,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,TEMP_29
30,한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp,TEMP_30


In [334]:
df2.columns

Index(['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일',
       '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '텍스트', '텍스트길이'],
      dtype='str')

In [335]:
df2.info() 

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   공고 번호      100 non-null    str    
 1   공고 차수      82 non-null     float64
 2   사업명        100 non-null    str    
 3   사업 금액      100 non-null    float64
 4   발주 기관      100 non-null    str    
 5   공개 일자      100 non-null    str    
 6   입찰 참여 시작일  100 non-null    str    
 7   입찰 참여 마감일  93 non-null     str    
 8   사업 요약      100 non-null    str    
 9   파일형식       100 non-null    str    
 10  파일명        100 non-null    str    
 11  텍스트        100 non-null    str    
 12  텍스트길이      7 non-null      float64
dtypes: float64(3), str(10)
memory usage: 1.5 MB


In [336]:
# 공고 차수랑 입찰 참여 마감일은 널값으로 두는게 낫다고함

In [337]:
df2 = df2.drop(columns=["텍스트길이"])

In [338]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   공고 번호      100 non-null    str    
 1   공고 차수      82 non-null     float64
 2   사업명        100 non-null    str    
 3   사업 금액      100 non-null    float64
 4   발주 기관      100 non-null    str    
 5   공개 일자      100 non-null    str    
 6   입찰 참여 시작일  100 non-null    str    
 7   입찰 참여 마감일  93 non-null     str    
 8   사업 요약      100 non-null    str    
 9   파일형식       100 non-null    str    
 10  파일명        100 non-null    str    
 11  텍스트        100 non-null    str    
dtypes: float64(2), str(10)
memory usage: 1.5 MB


In [339]:
# 저장
output_path = "/home/bidcoin/data_cleaning_final.csv"  
df2.to_csv(output_path, index=False, encoding="utf-8")

print(f"저장 완료: {output_path}")

저장 완료: /home/bidcoin/data_cleaning_final.csv
